# Qwen3.5-27B associative inference with thinking on (task 3)

This notebook runs associative inference on the 27B with Qwen's thinking mode, using the same thinking setup as the repo's other tasks. In the repo's earlier thinking runs (recognition, paired associates and serial order), thinking helped only on serial order, the task that needs reasoning over more than one memory. Associative inference is the clearest multi-hop memory task in the suite (A→B, B→C, so A→C), and the 27B is the only model that can do its parts. This notebook runs the 27B's gate cells with thinking on and compares every trial with its thinking-off twin.

**What runs:** associative inference on the 27B, cue_turn layout, both pair types, at 2, 5 and 25 chains, 50 trials per cell, all three probes by default (`PROBES` in the settings): the same cells, seed and episodes as the thinking-off run of 10/20.

**The thinking is the repo's own setup.** The `thinking` block from `configs/recognition_thinking_brady.yaml` (Qwen's recommended sampling: temperature 0.6, top_p 0.95, top_k 20, seeded per trial as `seed:trial`; budget 12,288 tokens) and `think()` from `bench/models.py`. The repo's associative-inference loop does not call `think()` yet, so `runpod/run_bench_ext.py` calls it per trial exactly the way the recognition loop does: one trace, then the forced 1-or-2 choice read off the logits after `</think>`, and the free reply written after the same trace. The repo's per-cell thinking metrics (`thinking_mean_tokens`, `thinking_frac_budget`, ...) are in every result file. The trace sampling uses its own random stream, so the episodes stay identical to the thinking-off run. The sides are not counterbalanced here, on purpose, for the trial-by-trial comparison; the report gives bias-corrected accuracy for both.

**The question it answers:** does thinking help the inference step, and in particular, why does the 27B fail inference at 2 chains while recalling the pairs at 93–98%? The traces of the 2-chain inference trials show what the model takes "indirectly associated" to mean.

**GPU savings built in:**

| Saving | Why it is safe | Effect |
|---|---|---|
| Loop stop (`LOOP_STOP`) | the only difference from the repo's thinking runs: the same sampled trace, cut once its last stretch repeats three times in a row, then closed and scored like a budget trace (recorded as stop `loop`). In the repo's earlier thinking runs, 14 of the 16 traces that hit the budget were loops, lines repeated up to 528 times. Set `LOOP_STOP = False` to match the repo's thinking runs exactly | a looping trial stops after ~100–700 tokens instead of 12,288 |
| Two queues on one card, when it has the room | this run is token-by-token decoding, which leaves a big GPU mostly idle; two copies of the 27B (~58 GB each) fit on an H200 | ~1.6–2× faster on an H200 |
| One layout | the comparison only needs the cells the thinking-off run has | 3× fewer trials than all layouts |
| `PROBES = ["inference"]` (optional) | the direct probes are at 93–100% with thinking off, so they say least; but dropping them is less like the other tasks' setup, so it is off by default | 3× fewer trials |

## Which pod

**Recommended:** one **H200** (CUDA 12.8+, 100 GB+ volume), where the notebook runs two queues side by side. An A100 80 GB also works (one queue), but it decodes at about half the speed, so it takes 3–4× longer for about the same money. If you just ran the long-context notebook on an H200, use the same pod or volume: the environment and weights are already there.

**Upload to `/workspace`:** the notebook, `qwen35-bench-master.zip`, and **`qwen35_runpod_results_assoc_20261020_1921.zip`**: it holds the 27B's thinking-off cells this run is compared with. Without it, the notebook runs those 18 cells first (about 20 minutes).

**Time on an H200 with two queues** (rough; the preflight measures it): all three probes ~15–20 h, inference only ~5–7 h. Almost all of it is the model writing its thinking, one token at a time.

## 1. Settings

In [ ]:
import os
from pathlib import Path

WORK = Path(os.environ.get("BENCH_WORK", "/workspace"))   # the pod's persistent volume
REPO = WORK / "qwen35-bench"                      # the repo is unpacked here
REPO_ZIP = WORK / "qwen35-bench-master.zip"       # upload the zip here...
GIT_URL = ""                                      # ...or clone instead: "https://github.com/<owner>/<repo>.git"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "") # only for cloning a private repo; never paste a token here
HF_TOKEN = os.environ.get("HF_TOKEN", "")         # optional: the Qwen3.5 checkpoints are public

CHAINS = [2, 5, 25]      # the thinking-off run's sizes, so every trial has a twin
PAIR_TYPES = ["image", "word"]
N_TRIALS = 50
PROBES = ["inference", "direct_ab", "direct_bc"]   # all three probes. Cells already finished (restored from an
                         # uploaded qwen35_runpod_results_*.zip) are skipped, so this only runs what is missing
THINK_BUDGET = 12288     # the repo's budget (configs/recognition_thinking_brady.yaml); the sampling is fixed in the config
LOOP_STOP = False        # False = exactly the repo's thinking runs: a looping trace runs to the budget and counts in the
                         # thinking_frac_budget (the loop-rate metric). True cuts loops early to save GPU time,
                         # but records them as stop "loop", which that metric does not count
PARALLEL = "auto"        # "auto": two queues if the GPU has 125 GB+ (an H200), else one; or 1 / 2
GPUS = "0"

CPU_THREADS = 0          # 0 = read the pod's real CPU limit
FORCE_MAIN = False       # True launches the full run even if the preflight failed (not recommended)
PRICE_PER_HOUR = 0.0     # your pod's $/hour, for the cost estimate (optional)

VENV = WORK / "envs" / "qwen35"
PY = VENV / "bin" / "python"
DATA = WORK / "memory_datasets"
HF_HOME = WORK / "hf_cache"
LOGS = WORK / "runpod_logs"
OUT = WORK / "outputs"
DOWNLOADS = WORK / "downloads"

Helpers used by the cells below. Nothing to edit; run once per kernel session (and again after a kernel restart).

In [ ]:
MIN_GPU_GB = 75
GPU_NOTE = "enough for one queue."

import json, re, shutil, signal, subprocess, sys, time, urllib.request, zipfile
from datetime import datetime
from IPython.display import Image, Markdown, clear_output, display

ENV = dict(os.environ, HF_HOME=str(HF_HOME), PYTHONUNBUFFERED="1",
           TRITON_CACHE_DIR=str(WORK / ".triton"), UV_CACHE_DIR=str(WORK / ".uv_cache"),
           UV_PYTHON_INSTALL_DIR=str(WORK / ".uv_python"), UV_HTTP_TIMEOUT="300")
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
# HF_HUB_OFFLINE: the jobs read weights from the local cache only. expandable_segments: the allocator
# grows segments instead of reserving new blocks, which avoids out-of-memory from fragmentation when
# every trial is a ~200k-token prompt of a slightly different size.
RUN_ENV = {"HF_HUB_OFFLINE": "1", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
for d in (LOGS, OUT, HF_HOME, DOWNLOADS):
    d.mkdir(parents=True, exist_ok=True)


def cpu_quota():
    """Cores this pod may actually use (its cgroup CPU limit), or None if there is no limit."""
    try:
        quota, period = Path("/sys/fs/cgroup/cpu.max").read_text().split()[:2]
    except (OSError, ValueError):
        try:
            quota = Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text().strip()
            period = Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text().strip()
        except OSError:
            return None
    if quota == "max" or int(quota) <= 0:
        return None
    return int(quota) / int(period)


VISIBLE_CPUS = os.cpu_count() or 1
CPU_QUOTA = cpu_quota()
THREADS = CPU_THREADS or max(1, int(min(VISIBLE_CPUS, CPU_QUOTA or VISIBLE_CPUS)))


def set_threads(n_queues):
    """Give each concurrent queue its share of the CPU (too many threads makes jobs several times slower)."""
    per = str(max(1, THREADS // n_queues))
    RUN_ENV.update(OMP_NUM_THREADS=per, MKL_NUM_THREADS=per)
    return per


set_threads(1)
print(f"CPU: {VISIBLE_CPUS} cores visible, " + (f"{CPU_QUOTA:.1f} usable" if CPU_QUOTA else "no limit")
      + f"; jobs use {THREADS} threads")

MODEL = "Qwen/Qwen3.5-27B"
WEIGHTS_GB = 55
EXT = "runpod/run_bench_ext.py"


def gpu_memory():
    """{GPU index: GB}, decimal GB like the result files' peak_gpu_gb. Empty without nvidia-smi."""
    smi = shutil.which("nvidia-smi")
    if not smi:
        return {}
    out = subprocess.run([smi, "--query-gpu=index,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    mem = {}
    for line in out.strip().splitlines():
        try:
            index, mib = [x.strip() for x in line.split(",")]
            mem[int(index)] = float(mib) * 2**20 / 1e9
        except ValueError:
            pass
    return mem


def capacity(gpus=None):
    mem = gpu_memory()
    return sum(mem.get(int(i), 0) for i in str(gpus or GPUS).split(","))


def run(cmd, cwd=None, check=True, echo=True):
    """Run a command, streaming its output into the cell. Returns the exit code if non-zero, else None."""
    shell = isinstance(cmd, str)
    if not shell:
        cmd = [str(c) for c in cmd]
    if echo:
        print("$", cmd if shell else " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, shell=shell, cwd=cwd, env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc or None


def bench(*args, check=True):
    """Run the benchmark environment's Python from the repo root."""
    return run([PY, *args], cwd=REPO, check=check, echo=False)


def gpu_usage():
    smi = shutil.which("nvidia-smi")
    if not smi:
        return
    out = subprocess.run([smi, "--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    for line in out.strip().splitlines():
        fields = [x.strip() for x in line.split(",")]
        if len(fields) < 4:
            continue
        i, util, used, total = fields[:4]
        power = f", {fields[4]} W" if len(fields) > 4 else ""
        print(f"GPU {i}: {util}% busy, {float(used) / 1024:.0f} of {float(total) / 1024:.0f} GiB in use{power}")


# ---- background job queues --------------------------------------------------------------------
_QUEUES = {}


def _proc_fields(pid):
    try:
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[:3]
    except (OSError, IndexError):
        return None


def queue_running(name):
    p = _QUEUES.get(name)
    if p is not None:
        return p.poll() is None
    pid_file = LOGS / f"{name}.pid"
    if not pid_file.exists():
        return False
    pid = int(pid_file.read_text())
    fields = _proc_fields(pid)
    if fields is None or fields[0] in ("Z", "X"):
        return False
    try:
        return b"run_queue.py" in Path(f"/proc/{pid}/cmdline").read_bytes()
    except OSError:
        return False


def launch(name, jobs):
    """Run jobs one after another in the background. Each job is (config, gpus, entry script or None),
    optionally with a 4th element: a list of model names to run (default: every model in the config)."""
    jobs = [tuple(j) + (None,) * (4 - len(j)) for j in jobs]
    if queue_running(name):
        print(f"Queue '{name}' is already running; watch it with status('{name}').")
        return
    for config, *_ in jobs:
        if not (REPO / config).exists():
            raise FileNotFoundError(f"{config} is not in {REPO}")
    queue_file = LOGS / f"{name}.queue.json"
    queue_file.write_text(json.dumps({
        "name": name, "python": str(PY), "repo": str(REPO), "log_dir": str(LOGS), "env": RUN_ENV,
        "jobs": [{"config": c, "task": None, "gpus": str(g), "entry": e,
                  "extra_args": [a for m in (models or []) for a in ("--model", m)]}
                 for c, g, e, models in jobs]}, indent=2))
    (LOGS / f"{name}.status.json").unlink(missing_ok=True)
    log = open(LOGS / f"{name}.queue.log", "a")
    p = subprocess.Popen([str(PY), "runpod/run_queue.py", str(queue_file)], cwd=REPO, env=ENV,
                         stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    _QUEUES[name] = p
    (LOGS / f"{name}.pid").write_text(str(p.pid))
    print(f"Started queue '{name}' (pid {p.pid}). Jobs, in order:")
    for config, gpus, entry, models in jobs:
        print(f"  {entry or 'run_bench.py'} {config}   GPU {gpus}" + (f"   only {', '.join(models)}" if models else ""))


def status(name="main", tail=5):
    """Job states, cells done, the running job's last log lines, GPU use."""
    st_file = LOGS / f"{name}.status.json"
    running = queue_running(name)
    if not st_file.exists():
        print(f"Queue '{name}': " + ("starting..." if running else "never launched."))
        return
    st = json.loads(st_file.read_text())
    state = ("running" if running else "finished" if st.get("finished")
             else "stopped" if st.get("stopped") else "not running")
    print(f"Queue '{name}': {state}   started {st['started']}"
          + (f", finished {st['finished']}" if st.get("finished") else ""))
    for job in st["jobs"]:
        took = ""
        if job.get("start"):
            end = datetime.fromisoformat(job["end"]) if job.get("end") else datetime.now()
            took = str(end - datetime.fromisoformat(job["start"])).split(".")[0]
        print(f"  [{job['state']:8s}] {job['config']}   GPU {job['gpus']}   {took}")
    print()
    bench("runpod/progress.py", *dict.fromkeys(j["config"] for j in st["jobs"]), check=False)
    active = [j for j in st["jobs"] if j["state"] == "running"]
    if active and active[0].get("log") and Path(active[0]["log"]).exists():
        print(f"\nLast lines of {Path(active[0]['log']).name}:")
        for line in Path(active[0]["log"]).read_text(errors="replace").splitlines()[-tail:]:
            print("   ", line[:220])
        if not running:
            print("\nThe queue process is gone but a job is still marked running (pod restart, or stop()).\n"
                  "Launch the same queue again: finished cells are kept and skipped.")
    for j in st["jobs"]:
        if j["state"] == "failed":
            print(f"\nFAILED: {j['config']} (exit code {j['returncode']}). End of its log, {Path(j['log']).name}:")
            for line in Path(j["log"]).read_text(errors="replace").splitlines()[-8:]:
                print("   ", line[:220])
    gpu_usage()


def wait(*names, every=60):
    """Refresh status() for the named queues until all finish. Interrupting only stops watching."""
    names = names or ("main",)
    try:
        while any(queue_running(n) for n in names):
            clear_output(wait=True)
            for n in names:
                status(n)
                print()
            print(f"Refreshing every {every} s. Interrupt this cell any time; the jobs keep running.")
            time.sleep(every)
    except KeyboardInterrupt:
        print("Stopped watching. The jobs keep running; check them with status().")
        return
    clear_output(wait=True)
    for n in names:
        status(n)
        print()


def _group_alive(pgid):
    for stat in Path("/proc").glob("[0-9]*/stat"):
        try:
            state, _, group = stat.read_text().rsplit(")", 1)[1].split()[:3]
        except (OSError, IndexError, ValueError):
            continue
        if group == str(pgid) and state not in ("Z", "X"):
            return True
    return False


def stop(name="main"):
    """Stop a queue and the job it is running. Finished cells are kept; launching again resumes."""
    if not queue_running(name):
        print(f"Queue '{name}' is not running.")
        return
    pgid = int((LOGS / f"{name}.pid").read_text())
    os.killpg(pgid, signal.SIGTERM)
    for _ in range(40):
        if not _group_alive(pgid):
            break
        time.sleep(0.5)
    else:
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    if name in _QUEUES:
        _QUEUES.pop(name).poll()
    st_file = LOGS / f"{name}.status.json"
    if st_file.exists():
        st = json.loads(st_file.read_text())
        now = datetime.now().isoformat(timespec="seconds")
        for j in st["jobs"]:
            if j["state"] == "running":
                j.update(state="stopped", end=now)
        st["stopped"] = now
        st_file.write_text(json.dumps(st, indent=2))
    print(f"Stopped queue '{name}'.")


def progress(config):
    """{done, failed, skipped, todo, total} for one config, without loading a model."""
    r = subprocess.run([str(PY), "runpod/progress.py", "--json", config], cwd=REPO, env=ENV,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout + r.stderr)
        return None
    return json.loads(r.stdout)[0]


print("Helpers ready.")

## 2. GPU, driver, disk and RAM

In [ ]:
smi = shutil.which("nvidia-smi")
if smi is None:
    raise RuntimeError("nvidia-smi not found: this pod has no visible GPU.")
out = subprocess.run([smi, "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True, check=True).stdout
GPU_MEM, driver = {}, "0"
for line in out.strip().splitlines():
    idx, name, mem, driver = [x.strip() for x in line.split(",")]
    GPU_MEM[int(idx)] = float(mem) * 2**20 / 1e9
    print(f"GPU {idx}: {name}, {float(mem) / 1024:.0f} GiB = {GPU_MEM[int(idx)]:.1f} GB, driver {driver}")

fatal, warnings = [], []
if int(driver.split(".")[0]) < 570:
    fatal.append(f"Driver {driver} is older than 570; torch 2.10 with CUDA 12.8 needs 570+. "
                 "Deploy a new pod filtered to CUDA 12.8 or newer.")
missing = [g for g in GPUS.split(",") if int(g) not in GPU_MEM]
if missing:
    fatal.append(f"GPU {','.join(missing)} is in the settings but not on this pod.")
if capacity() < 75:
    fatal.append(f"The 27B needs ~60 GB even at short prompts; GPUS={GPUS!r} gives {capacity():.0f} GB.")
elif capacity() < MIN_GPU_GB:
    warnings.append(f"{capacity():.0f} GB of GPU memory: {GPU_NOTE}")

weights_here = (HF_HOME / "hub" / ("models--" + MODEL.replace("/", "--"))).exists()
need = (0 if weights_here else WEIGHTS_GB) + (0 if PY.exists() else 12) + 5
free = shutil.disk_usage(WORK).free / 1e9
print(f"\nDisk: {free:.0f} GB free in {WORK}; about {need:.0f} GB still needed (weights, environment, headroom)")
if free < need:
    fatal.append(f"Not enough disk in {WORK}: {free:.0f} GB free, about {need:.0f} GB needed. Use a 100 GB+ volume.")
if weights_here and PY.exists():
    print("The environment and the 27B weights are already on this volume: setup will take a few minutes.")

ram = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1)) / 1e6
for limit_file in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
    try:
        value = Path(limit_file).read_text().strip()
        if value.isdigit():
            ram = min(ram, int(value) / 1e9)
    except OSError:
        pass
print(f"RAM: {ram:.0f} GB")
if ram < 32:
    warnings.append(f"Only {ram:.0f} GB of RAM; 32 GB+ is safer (thousands of images are decoded per prompt).")

print()
for w in warnings:
    print("WARNING:", w)
if fatal:
    raise RuntimeError("Fix before continuing:\n  - " + "\n  - ".join(fatal))
print("Checks passed." if not warnings else "No blocking problems.")

## 3. Get the repo, and restore earlier results

In [ ]:
if (REPO / "run_bench.py").exists():
    print(f"Repo already at {REPO}")
elif GIT_URL:
    url = GIT_URL
    if GITHUB_TOKEN and url.startswith("https://"):
        url = url.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1)
    print(f"Cloning {GIT_URL} ...")
    run(["git", "clone", "--depth", "1", url, REPO], echo=False)     # not echoed: the URL may hold a token
elif REPO_ZIP.exists():
    with zipfile.ZipFile(REPO_ZIP) as z:
        top = z.namelist()[0].split("/")[0]
        z.extractall(WORK)
        commit = z.comment.decode(errors="replace").strip()
    (WORK / top).rename(REPO)
    (REPO / "RUNPOD_SOURCE.json").write_text(json.dumps(
        {"source": REPO_ZIP.name, "archive_commit": commit or None,
         "unpacked": datetime.now().isoformat(timespec="seconds")}, indent=2))
else:
    raise FileNotFoundError(f"No repo yet. Upload qwen35-bench-master.zip to {WORK} "
                            "(drag it into the Jupyter file browser), or set GIT_URL in the settings cell.")

for p in ("run_bench.py", "bench/runner.py", "bench/models.py", "bench/tasks/associative_inference_eval.py",
          "requirements.txt", "requirements.lock.txt"):
    assert (REPO / p).exists(), f"{p} is missing from the repo"
if (REPO / "RUNPOD_SOURCE.json").exists():
    print("Unpacked from", json.loads((REPO / "RUNPOD_SOURCE.json").read_text()))

Copies the finished result files from any `qwen35_runpod_results_*.zip` in `/workspace` into the repo, so those cells are skipped (files already in the repo are never overwritten).

In [ ]:
previous = sorted(WORK.glob("qwen35_runpod_results_*.zip"))
for zip_path in previous:
    restored = kept = 0
    try:
        with zipfile.ZipFile(zip_path) as z:
            for name in z.namelist():
                if not (name.startswith("results/") and name.endswith(".json")):
                    continue
                if (REPO / name).exists():
                    kept += 1
                else:
                    z.extract(name, REPO)
                    restored += 1
    except zipfile.BadZipFile:
        print(f"{zip_path.name} is not a complete zip (the upload may have been cut off); upload it again.")
        continue
    print(f"{zip_path.name}: restored {restored} result files" + (f", kept {kept} already here" if kept else ""))
if not previous:
    print(f"No results zip in {WORK}.")

## 4. Python environment

The repo's pins (`setup_env.sh`, `requirements.lock.txt`) in a Python 3.12 environment on the volume. On a fresh volume this takes 5–15 minutes; on a volume that already has it, seconds. The one slow download (`causal-conv1d`, from GitHub) is fetched once into `/workspace/downloads`, resumable, and installed from there.

In [ ]:
CAUSAL_CONV1D_WHEEL = ("https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/"
                       "causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl")
CHECK = ("import torch, transformers, fla, causal_conv1d; assert torch.cuda.is_available(), 'CUDA not available'; "
         "print('torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| GPU', torch.cuda.get_device_name(0))")


def env_ok():
    if not PY.exists():
        return False
    r = subprocess.run([str(PY), "-c", CHECK], env=ENV, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    return r.returncode == 0


def fetch_wheel(url, attempts=4):
    """Download once to the volume, resuming if cut off; uv then installs from the file. Installing from
    the URL made uv download it twice (to resolve, then to install), 40 minutes on a slow pod."""
    target = DOWNLOADS / url.rsplit("/", 1)[1].replace("%2B", "+")
    if target.exists() and zipfile.is_zipfile(target):
        print(f"Using the copy already on the volume: {target.name}")
        return target
    for attempt in range(1, attempts + 1):
        rc = run(["curl", "-L", "--fail", "--retry", "5", "--retry-delay", "5", "--connect-timeout", "30",
                  "-C", "-", "-o", target, url], check=False)
        if rc is None and zipfile.is_zipfile(target):
            return target
        print(f"  attempt {attempt} of {attempts} did not finish; retrying", flush=True)
        if target.exists() and not zipfile.is_zipfile(target) and attempt > 1:
            target.unlink()
        time.sleep(10 * attempt)
    raise RuntimeError(f"Could not download {url}")


if env_ok():
    print("Environment ready:", VENV)
else:
    t0 = time.time()
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv_bin = Path(sys.executable).parent / "uv"
    uv = [str(uv_bin)] if uv_bin.exists() else [sys.executable, "-m", "uv"]
    wheel = fetch_wheel(CAUSAL_CONV1D_WHEEL)        # the slow download first, while nothing else waits on it
    run([*uv, "venv", "--python", "3.12", VENV])
    run([*uv, "pip", "install", "--python", PY, "torch==2.10.0", "torchvision==0.25.0",
         "--index-url", "https://download.pytorch.org/whl/cu128"])
    run([*uv, "pip", "install", "--python", PY, wheel])
    lock = (REPO / "requirements.lock.txt").read_text().splitlines()
    pins = [l for l in lock if l.strip() and not re.match(r"(torch|torchvision|triton|nvidia-|cuda-|causal-conv1d)\b", l)]
    (WORK / "constraints.txt").write_text("\n".join(pins) + "\n")
    run([*uv, "pip", "install", "--python", PY, "-r", REPO / "requirements.txt", "-c", WORK / "constraints.txt"])
    assert env_ok(), "The environment check failed; read the output above."
    print(f"Environment ready: {VENV} ({(time.time() - t0) / 60:.0f} min)")

## 5. Stimuli

The Brady 2008 objects and the word pool, from the public `csiyer/vision-memory` repo.

In [ ]:
STIMULI = {
    "memory_datasets_brady.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_brady.zip", 80955797),
    "memory_datasets_mst.zip": ("https://raw.githubusercontent.com/csiyer/vision-memory/main/memory_datasets_mst.zip", 63189250),
}
EXPECTED = {"Brady2008Objects": 2400}


def download(url, target, size, attempts=4):
    part = target.with_name(target.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            urllib.request.urlretrieve(url, part)
        except OSError as e:
            print(f"  attempt {attempt} of {attempts} failed ({e.__class__.__name__}); retrying", flush=True)
            time.sleep(5 * attempt)
            continue
        if part.stat().st_size != size:
            raise RuntimeError(f"{target.name} is {part.stat().st_size} bytes, expected {size}: "
                               "the source repo may have changed, so check the stimuli before running.")
        part.replace(target)
        return
    raise RuntimeError(f"Could not download {url} after {attempts} attempts.")


def stimuli_report():
    folder = DATA / "Brady2008Objects"
    count = sum(1 for p in folder.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")) if folder.is_dir() else 0
    pool = DATA / "wasnorm_wordpool.txt"
    words = len(pool.read_text().split()) if pool.exists() else 0
    return count == 2400 and words >= 1637, count, words


ok, *_ = stimuli_report()
if not ok:
    for name, (url, size) in STIMULI.items():
        target = DOWNLOADS / name
        if not (target.exists() and target.stat().st_size == size):
            print(f"Downloading {name} ({size / 1e6:.0f} MB) ...", flush=True)
            download(url, target, size)
        with zipfile.ZipFile(target) as z:
            z.extractall(WORK)
ok, count, words = stimuli_report()
print(f"Brady2008Objects: {count} images (expected 2400); word pool: {words} words (expected 1638)")
assert ok, "The stimuli are incomplete; see the counts above."

link = REPO / "memory_datasets"
if link.is_symlink() and link.resolve() != DATA.resolve():
    link.unlink()
if not link.exists() and not link.is_symlink():
    link.symlink_to(DATA)
print(f"{link} -> {link.resolve()}")

## 6. Model weights

Only the 27B, about 55 GB, cached in `/workspace/hf_cache`.

In [ ]:
print(f"== {MODEL} (~{WEIGHTS_GB} GB; files already in the cache are skipped)", flush=True)
bench("-c", f"from huggingface_hub import snapshot_download; print(snapshot_download({MODEL!r}))")
print("Weights ready.")

In [ ]:
def twins_here():
    folder = REPO / "results/assoc_inf_ab/associative_inference"
    files = folder.glob("qwen3.5-27b__*prompt-cue_turn__seed0.json") if folder.exists() else []
    return [f for f in files if any(f"n_chains-{n}_" in f.name for n in CHAINS)
            and any(f"pair_type-{pt}_" in f.name for pt in PAIR_TYPES)]      # only the cells this run pairs with

n_twins = len(twins_here())
print(f"The 27B's thinking-off cells (cue_turn) here: {n_twins} of {3 * len(CHAINS) * len(PAIR_TYPES)}.")
if n_twins < 3 * len(CHAINS) * len(PAIR_TYPES):
    print("The missing ones run first in the main queue (~20 minutes). To skip that, upload "
          "qwen35_runpod_results_assoc_20261020_1921.zip to /workspace and run section 3 again.")

## 7. Scripts and configs

Written into the repo under `runpod/` and `configs/runpod/`; no existing repo file is modified. `run_bench_ext.py` is `run_bench.py` plus `scoring: think`, which calls the repo's `think()` once per trial for associative inference. The configs are generated from the settings by `make_configs.py`; they carry the repo's `thinking` block.

In [ ]:
(REPO / "configs" / "runpod").mkdir(parents=True, exist_ok=True)
(REPO / "runpod").mkdir(parents=True, exist_ok=True)
print("ok")

In [ ]:
%%writefile {REPO}/runpod/run_bench_ext.py
#!/usr/bin/env python
"""run_bench.py, plus opt-in scoring modes and GPU-saving options, set per model in the config.

Usage (from the repo root): python runpod/run_bench_ext.py <config.yaml> [run_bench.py's options]

The task code is untouched: everything here wraps the evaluator or the task adapter. A model entry
with none of the keys below runs exactly as under run_bench.py, so one config can mix both.

Scoring (`scoring:`)
  standard            The forced choice after "Answer: ", as in run_bench.py.
  reason_then_choose  First a free reply (greedy, thinking off, up to `max_new_tokens`), then the forced
                      choice with that reply in context: "<reply>\n\nAnswer: ".
  think               The repo's thinking mode, for a task whose own loop does not call think() (associative
                      inference): the model entry's `thinking:` block (as in configs/*_thinking_*.yaml) is
                      passed to the evaluator, and each trial runs bench.models' think() with the trial
                      seed "<seed>:<trial>", then the forced choice and the free reply both read after that
                      one trace, exactly as afc_recognition_eval does. Same sampling, budget and closing.
                      One addition, `loop_stop` (default true): the trace is the same sampled trace, but it
                      is cut as soon as its last stretch repeats itself three times over (earlier runs had lines
                      repeated up to 528 times; such a trace would otherwise run on to the budget). It is
                      then closed and scored like a budget trace, and recorded with stop "loop".

Per-trial stats in both reasoning modes: reasoning_tokens, reasoning_finished, stated_answer, and for
thinking thinking_end (closed | loop | budget | eos) and the repo's own `thinking` record.

GPU-saving and bookkeeping options
  free_text_trials: K     Generate the free-text reply only for the first K trials of each cell; later
                          trials get free_response None and free_text_skipped true. The score is the
                          forced choice and never reads the reply, which is only a readability check, so
                          K = 5 keeps that check and saves one full prefill per trial (about half the
                          time at long context, where each prefill is the whole study list).
  skip_capacity_probe     Do not run the per-cell capacity probe (one extra full-size request per cell).
                          Use when a preflight has already run the largest cells: an out-of-memory trial
                          is still recorded as an error. Always skipped with thinking on (the probe
                          has no trace to pass, so the evaluator refuses it).
  counterbalance_sides    Associative inference only: put the correct option on side 1 in exactly half of
                          each cell's trials and on side 2 in the other half (shuffled), by swapping the
                          two options where needed. Episodes, probes and foils are unchanged. With a side
                          bias (the 27B answers "2" on ~70% of inference trials), unbalanced sides move raw
                          accuracy; balanced sides make raw and bias-corrected accuracy agree.

Always recorded with this entry point: prompt_tokens (the trial's prompt length, the x-axis for
context-length plots), score_seconds and free_text_seconds (per-trial timing, for estimates), a context
guard (a prompt longer than the model's max_position_embeddings is an error, not a silent
extrapolation), and balanced_accuracy in each cell's metrics for 2-choice tasks.
"""
import hashlib
import json
import random
import re
import sys
import time
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

ANSWER_SEPARATOR = "\n\n"
THINK_OPEN, THINK_CLOSE = "<think>", "</think>"

# A conclusion: "the correct digit is 2", "Answer: **1**", "I choose image 2", or "Image 2 is the
# correct answer". The last one in the reply counts, and a bare "1"/"2" reply counts too. Descriptions
# such as "Image 1 shows a teddy bear" are deliberately not matched, so a reply cut off before its
# conclusion has no stated answer. Only a diagnostic: the scored answer is always the forced choice.
_CONCLUSIONS = (
    re.compile(r"\b(?:answer|digit|choice|correct|pick|choose|chose|select)\w*\s*(?:is|would\s+be)?\s*[:=]?\s*"
               r"(?:\*\*)?\s*(?:the\s+)?(?:image|word|option|number)?\s*\(?\s*(?:\*\*)?([12])\b", re.IGNORECASE),
    re.compile(r"\b([12])\b\W{0,6}(?:is|would\s+be)\s+(?:the\s+)?(?:correct|right|answer)", re.IGNORECASE),
    re.compile(r"\b(?:therefore|thus|hence|so)\b[^.\n]{0,150}?\b(?:is|would\s+be)\s+(?:\*\*)?\s*(?:the\s+)?"
               r"(?:image|word|option|number)?\s*\(?\s*(?:\*\*)?([12])\b", re.IGNORECASE),
)


def stated_answer(reply):
    """The answer a free reply states, "1" or "2", or None if it states none."""
    reply = (reply or "").replace(THINK_OPEN, "").replace(THINK_CLOSE, "")
    text = reply.strip().strip("*").rstrip(".").strip()
    if text in ("1", "2"):
        return text
    found = [(m.start(1), m.group(1)) for pattern in _CONCLUSIONS for m in pattern.finditer(reply)]
    return max(found)[1] if found else None


def is_looping(ids, windows=(8, 16, 32, 64, 128, 256), repeats=3):
    """True if the sequence ends in some stretch of `w` tokens repeated `repeats` times back to back."""
    for w in windows:
        if len(ids) < w * repeats:
            break
        tail = ids[-w:]
        if all(ids[-(k + 1) * w:-k * w] == tail for k in range(1, repeats)):
            return True
    return False


# ---------------------------------------------------------------------------------------------------
# Evaluator mixins. The per-trial log is `_ext_log`, not `trial_log`, so the repo's own runner hooks for
# its ported reason_then_choose (bench/reasoned.py: reset/attach on `trial_log`) leave these runs alone. Order in the composed class: Extras, then the scoring mixin, then Generate, then
# the real evaluator (bench.models.Qwen35Evaluator, or a fake in tests). None of them defines __init__;
# ext_setup() sets their state after the evaluator has loaded. Every override passes on the keyword
# arguments it does not use (bench.models passes `thought=` and `opening_think=`).

class ExtrasMixin:
    """Per-trial bookkeeping (one trial = one score_choices call) and the GPU-saving options."""

    def _ext_begin_cell(self, seed=0):
        self._ext_log.clear()
        self._trial_index = 0
        self._cell_seed = seed

    def check_image_capacity(self, n_images, sample_image=None):
        if self.skip_capacity_probe or getattr(self, "thinking", None):
            return True
        return super().check_image_capacity(n_images, sample_image=sample_image)

    def _prepare_inputs(self, messages, prefill=None, **kwargs):
        inputs = super()._prepare_inputs(messages, prefill, **kwargs)
        n = int(inputs["input_ids"].shape[1])
        # The first prompt of a trial is the bare prompt (with thinking, the one think() opens).
        if self._trial_extra is not None and "prompt_tokens" not in self._trial_extra:
            self._trial_extra["prompt_tokens"] = n
        if self.max_context and n > self.max_context:
            raise ValueError(f"prompt of {n} tokens exceeds the model's context window ({self.max_context})")
        return inputs

    def score_choices(self, messages, choices=("1", "2"), prefill="Answer: ", **kwargs):
        self._trial_extra = {}
        self._last_scored = messages
        t0 = time.perf_counter()
        try:
            forced = super().score_choices(messages, choices=choices, prefill=prefill, **kwargs)
        finally:
            extra, self._trial_extra = self._trial_extra, None
        extra["score_seconds"] = round(time.perf_counter() - t0, 3)
        self._ext_log.append(extra)
        self._trial_index += 1
        return forced

    def _call_api(self, messages, **kwargs):
        cached = getattr(self, "_reply_cache", None)
        has_cached = cached is not None and cached[0] is messages
        if (not has_cached and messages is self._last_scored and self.free_text_trials is not None
                and self._trial_index > self.free_text_trials):
            self._ext_log[-1]["free_text_skipped"] = True
            return None
        t0 = time.perf_counter()
        reply = super()._call_api(messages, **kwargs)
        if messages is self._last_scored and self._ext_log and not has_cached:
            self._ext_log[-1]["free_text_seconds"] = round(time.perf_counter() - t0, 3)
        return reply

    def _note(self, **extra):
        if self._trial_extra is not None:
            self._trial_extra.update(extra)


class ReasonThenChooseMixin:
    """A free reply first, then the forced choice read after it. The task's free-text call gets that
    same reply back, so no trial is generated twice."""

    def score_choices(self, messages, choices=("1", "2"), prefill="Answer: ", **kwargs):
        reply, n_tokens, finished = self._generate(messages)
        self._reply_cache = (messages, reply)
        reasoned_prefill = (reply + ANSWER_SEPARATOR + prefill) if reply else prefill
        forced = super().score_choices(messages, choices=choices, prefill=reasoned_prefill, **kwargs)
        self._note(reasoning_tokens=n_tokens, reasoning_finished=finished, stated_answer=stated_answer(reply))
        return forced

    def _call_api(self, messages, **kwargs):
        cached = self._reply_cache
        if cached is not None and cached[0] is messages:
            self._reply_cache = None
            return cached[1]
        return super()._call_api(messages, **kwargs)


class ThinkMixin:
    """bench.models' thinking mode for a task that does not call think() itself: one trace per trial,
    which the forced choice and the free reply both follow (as afc_recognition_eval does)."""

    def score_choices(self, messages, choices=("1", "2"), prefill="Answer: ", thought=None, **kwargs):
        if thought is None:
            thought = self.think(messages, f"{self._cell_seed}:{self._trial_index}")
        self._thought = (messages, thought)
        forced = super().score_choices(messages, choices=choices, prefill=prefill, thought=thought, **kwargs)
        self._note(thinking={k: thought[k] for k in ("trace", "n_tokens", "stop", "seed")},
                   reasoning_tokens=thought["n_tokens"], reasoning_finished=thought["stop"] == "closed",
                   thinking_end=thought["stop"], stated_answer=stated_answer(thought["trace"]))
        return forced

    def _call_api(self, messages, thought=None, **kwargs):
        held = getattr(self, "_thought", None)
        if thought is None and held is not None and held[0] is messages:
            thought = held[1]
        return super()._call_api(messages, thought=thought, **kwargs)

    def think(self, messages, trial_seed):
        """bench.models.Qwen35Evaluator.think, plus the loop stop. Same prompt, seed, sampling, stop
        tokens and closing, so up to a cut the trace is token for token the one it would sample."""
        if not self.thinking or not self.loop_stop:
            return super().think(messages, trial_seed)
        import torch
        from transformers import StoppingCriteria, StoppingCriteriaList

        cfg = self.thinking
        inputs = self._prepare_inputs(messages, opening_think=True)
        start = int(inputs["input_ids"].shape[1])

        class LoopStop(StoppingCriteria):
            looped = False

            def __call__(self, input_ids, scores, **kw):
                n = input_ids.shape[1] - start
                done = n >= 24 and n % 8 == 0 and is_looping(input_ids[0, start:].tolist())
                if done:
                    LoopStop.looped = True
                return torch.full((input_ids.shape[0],), done, dtype=torch.bool, device=input_ids.device)

        torch.manual_seed(int.from_bytes(hashlib.sha256(str(trial_seed).encode()).digest()[:8], "big"))
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=cfg["budget"], do_sample=True,
                temperature=cfg["temperature"], top_p=cfg["top_p"], top_k=cfg["top_k"],
                eos_token_id=[self.think_end_id, *self.eos_token_id], pad_token_id=self.pad_token_id,
                stopping_criteria=StoppingCriteriaList([LoopStop()]),
            )
        ids = out[0, start:].tolist()
        if self.think_end_id in ids:
            stop = "closed"
            ids = ids[:ids.index(self.think_end_id)]
        elif ids and ids[-1] in self.eos_token_id:
            stop = "eos"
            ids = ids[:-1]
        elif LoopStop.looped:
            stop = "loop"
        else:
            stop = "budget"
        trace = self.processor.tokenizer.decode(ids, skip_special_tokens=False)
        close = "</think>\n\n" if trace.endswith("\n") else "\n</think>\n\n"
        return {"trace": trace, "n_tokens": len(ids), "stop": stop, "seed": str(trial_seed),
                "prefill": trace + close}


class GenerateMixin:
    """The generation call reason_then_choose needs, on a transformers model."""

    def _generate(self, messages):
        """Qwen35Evaluator._call_api, also returning the token count and whether it ended by itself."""
        import torch

        inputs = self._prepare_inputs(messages)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=self.max_new_tokens, do_sample=False,
                eos_token_id=self.eos_token_id, pad_token_id=self.pad_token_id,
            )
        new = out[0, inputs["input_ids"].shape[1]:]
        n_tokens = int(new.shape[0])
        finished = n_tokens < self.max_new_tokens or (n_tokens > 0 and int(new[-1]) in self.eos_token_id)
        reply = self.processor.batch_decode(new.unsqueeze(0), skip_special_tokens=True)[0]
        return reply.strip(), n_tokens, finished


SCORERS = {"standard": (), "reason_then_choose": (ReasonThenChooseMixin,), "think": (ThinkMixin,)}


def evaluator_class(base, scoring="standard", generate=True):
    """The evaluator class for a config's model. generate=False leaves _generate to the base, for
    tests whose fake model supplies it."""
    if scoring not in SCORERS:
        raise ValueError(f"unknown scoring {scoring!r}; expected one of {sorted(SCORERS)}")
    gen = (GenerateMixin,) if generate else ()
    return type(f"Ext{base.__name__}", (ExtrasMixin, *SCORERS[scoring], *gen, base), {})


def ext_setup(evaluator, model_cfg):
    """The mixins' state, set once the evaluator exists."""
    evaluator._ext_log = []
    evaluator._trial_index = 0
    evaluator._cell_seed = 0
    evaluator._trial_extra = None
    evaluator._last_scored = None
    evaluator._reply_cache = None
    evaluator._thought = None
    evaluator.free_text_trials = model_cfg.get("free_text_trials")
    evaluator.skip_capacity_probe = bool(model_cfg.get("skip_capacity_probe", False))
    evaluator.counterbalance_sides = bool(model_cfg.get("counterbalance_sides", False))
    evaluator.loop_stop = bool(model_cfg.get("loop_stop", True))
    if model_cfg.get("scoring") == "think" and not getattr(evaluator, "thinking", None):
        raise ValueError("scoring: think needs a `thinking:` block with enabled: true in the model entry")
    model = getattr(evaluator, "model", None)
    config = getattr(model, "config", None)
    config = getattr(config, "text_config", config)
    evaluator.max_context = int(getattr(config, "max_position_embeddings", 0) or 0)
    return evaluator


# ---------------------------------------------------------------------------------------------------
# The task adapter wrapper: per-cell setup, side counterbalancing, and the per-trial stats.

def balanced_sides(n, key):
    """n targets, half 1 and half 2 (the odd one out drawn at random), shuffled reproducibly."""
    rng = random.Random(key)
    sides = [1, 2] * (n // 2) + ([rng.choice([1, 2])] if n % 2 else [])
    rng.shuffle(sides)
    return sides


def counterbalanced(make_probe, sides):
    """make_probe, with the two options swapped where needed so trial i's target is sides[i]."""
    calls = iter(range(len(sides)))

    def patched(trial_data, probe_type, rng):
        probe = make_probe(trial_data, probe_type, rng)
        want = sides[next(calls)]
        swapped = probe["target"] != want
        if swapped:
            key = "images" if "images" in probe else "options"
            probe = {**probe, key: list(reversed(probe[key])), "target": want}
        probe["metadata"] = {**probe.get("metadata", {}), "side_swapped": swapped}
        return probe

    return patched


def cell_metrics(trials):
    n = len(trials)
    if not n:
        return {}
    metrics = {}
    by_target = {}
    for t in trials:
        by_target.setdefault(t.get("target"), []).append(int(t.get("correct") or 0))
    if set(by_target) == {1, 2}:
        metrics["balanced_accuracy"] = sum(sum(v) / len(v) for v in by_target.values()) / 2
    tokens = [t["prompt_tokens"] for t in trials if "prompt_tokens" in t]
    if tokens:
        metrics["mean_prompt_tokens"] = sum(tokens) / len(tokens)
    if "reasoning_tokens" in trials[0]:
        stated = [t for t in trials if t.get("stated_answer") is not None]
        metrics.update({
            "mean_reasoning_tokens": sum(t["reasoning_tokens"] for t in trials) / n,
            "frac_reasoning_cut_off": sum(not t["reasoning_finished"] for t in trials) / n,
            "frac_stated_answer": len(stated) / n,
            # Of the trials whose reply states an answer, how often the forced choice read after it agrees.
            "stated_forced_agreement": (sum(str(t["stated_answer"]) == str(t.get("choice", t["reported"]))
                                            for t in stated) / len(stated)) if stated else None,
        })
    if "thinking_end" in trials[0]:
        for end in ("closed", "loop", "budget", "eos"):
            metrics[f"frac_thinking_{end}"] = sum(t["thinking_end"] == end for t in trials) / n
    if "thinking" in trials[0]:
        from bench.tasks.thinking import thinking_metrics
        metrics.update(thinking_metrics(trials))      # the repo's per-cell summary, as in its thinking runs
    skipped = sum(bool(t.get("free_text_skipped")) for t in trials)
    if skipped:
        metrics["frac_free_text_skipped"] = skipped / n
    return metrics


def wrap_adapter(adapter):
    """The task adapter, run with per-cell evaluator setup and the per-trial stats attached after."""

    class Wrapped:
        NAME = adapter.NAME

        def __getattr__(self, name):
            return getattr(adapter, name)

        @staticmethod
        def run(evaluator, condition, seed):
            if not hasattr(evaluator, "_ext_log"):
                return adapter.run(evaluator, condition, seed)
            evaluator._ext_begin_cell(seed)
            patch = None
            if getattr(evaluator, "counterbalance_sides", False):
                if not hasattr(adapter, "make_probe"):
                    raise ValueError(f"counterbalance_sides is only supported for associative_inference, "
                                     f"not {adapter.NAME}")
                key = "sides:" + json.dumps({"seed": seed, **condition}, sort_keys=True)
                patch = adapter.make_probe
                adapter.make_probe = counterbalanced(patch, balanced_sides(condition["n_trials"], key))
            try:
                result = adapter.run(evaluator, condition, seed)
            finally:
                if patch is not None:
                    adapter.make_probe = patch
            trials = result.get("trials") or []
            if trials:
                log = evaluator._ext_log
                if len(log) != len(trials):
                    raise RuntimeError(f"{len(log)} scored choices for {len(trials)} trials; "
                                       "this task does not score exactly one choice per trial")
                for trial, extra in zip(trials, log):
                    trial.update(extra)
                result["metrics"].update(cell_metrics(trials))
            return result

    return Wrapped()


def build_evaluator(model_cfg):
    """bench.models.build_evaluator, extended with the scoring modes and options above."""
    from bench import models

    import inspect

    scoring = model_cfg.get("scoring", "standard")
    cls = evaluator_class(models.Qwen35Evaluator, scoring)
    kwargs = dict(
        model_id=model_cfg["model_id"],
        max_image_size=model_cfg.get("max_image_size", 512),
        max_new_tokens=model_cfg.get("max_new_tokens", 100),
        layer_variant=model_cfg.get("layer_variant"),
        name=model_cfg["name"],
        device_map=model_cfg.get("device_map", "cuda"),
        thinking=model_cfg.get("thinking"),
    )
    # Newer versions of the repo take a per-card weight cap; pass it on only where it is accepted.
    if "max_memory" in inspect.signature(models.Qwen35Evaluator.__init__).parameters:
        kwargs["max_memory"] = model_cfg.get("max_memory")
    evaluator = ext_setup(cls(**kwargs), model_cfg)
    notes = [f"scoring={scoring}"]
    if scoring == "reason_then_choose":
        notes.append(f"reply up to {evaluator.max_new_tokens} tokens")
    if evaluator.thinking:
        t = evaluator.thinking
        notes.append(f"thinking on: budget {t['budget']}, temperature {t['temperature']}, top_p {t['top_p']}, "
                     f"top_k {t['top_k']}, loop_stop={evaluator.loop_stop}")
    if evaluator.free_text_trials is not None:
        notes.append(f"free text on the first {evaluator.free_text_trials} trials per cell")
    if evaluator.skip_capacity_probe:
        notes.append("capacity probe skipped")
    if evaluator.counterbalance_sides:
        notes.append("answer sides counterbalanced")
    print(f"{model_cfg['name']}: " + "; ".join(notes) + f"; context window {evaluator.max_context}", flush=True)
    return evaluator


def install():
    import bench.runner
    from bench.registry import get_adapter

    bench.runner.build_evaluator = build_evaluator
    bench.runner.get_adapter = lambda name: wrap_adapter(get_adapter(name))


if __name__ == "__main__":
    install()
    import run_bench

    run_bench.main()

In [ ]:
%%writefile {REPO}/runpod/make_configs.py
#!/usr/bin/env python
"""Write the configs for the two 27B associative-inference runs, from the notebook's settings.

Usage (from the repo root):
    python runpod/make_configs.py ladder --out configs/runpod/assoc_ladder_27b.yaml
        [--image 2,5,25,100,250,500,800] [--word 2,5,25,100,250,500,800,1200] [--trials 50]
        [--free-text-trials 5] [--preflight]
    python runpod/make_configs.py think --out configs/runpod/assoc_think_27b.yaml
        [--pair-types image,word] [--chains 2,5,25] [--probes inference,direct_ab,direct_bc] [--trials 50]
        [--budget 12288] [--no-loop-stop] [--preflight]
    python runpod/make_configs.py twins --out configs/runpod/assoc_inf_ab_27b_cue_turn.yaml
        [--pair-types image,word] [--chains 2,5,25] [--trials 50]

ladder: one task entry per (size, pair type), smallest size first and the two pair types interleaved,
so the runner finishes every probe at one size before starting the next: a run stopped early still
leaves a complete curve up to where it got. Layout cue_turn (the 27B passes the gate in all three), all
three probes, sides counterbalanced, free text only on the first trials of each cell, no capacity probe
(the preflight runs the largest cells). --preflight instead writes one inference trial per size, in the
order given (largest first), with free text on so the reply at full length can be read.

think: the standard gate cells for the 27B (cue_turn, the given chain counts and probes), with thinking
on exactly as in the repo's thinking configs (configs/recognition_thinking_brady.yaml): its `thinking`
block (budget 12,288, temperature 0.6, top_p 0.95, top_k 20, seeded per trial) and its think(), via
scoring: think in runpod/run_bench_ext.py, plus the loop stop. No side counterbalancing and the same
seed, so every trial pairs with its thinking-off twin in results/assoc_inf_ab. --preflight writes two
inference trials at the largest chain count per pair type, under a separate run name.

twins: those thinking-off cells themselves, exactly as in the repo's gate (configs/assoc_inf_ab.yaml, the
27B and cue_turn only), for when the earlier results zip is not at hand. Run with run_bench.py; cells
already restored are skipped.
"""
import argparse
from pathlib import Path

MODEL_27B = "  - name: qwen3.5-27b\n    model_id: Qwen/Qwen3.5-27B\n    max_image_size: 512\n    device_map: auto\n"


def ints(text):
    return [int(x) for x in text.split(",") if x.strip()]


def task_entry(pair_type, chains, probes, trials):
    return (f"  - task: associative_inference\n    datasets: [Brady2008]\n    pair_types: [{pair_type}]\n"
            f"    probe_types: [{', '.join(probes)}]\n    n_chains: [{', '.join(map(str, chains))}]\n"
            f"    prompts: [cue_turn]\n    n_trials: {trials}\n")


def ladder(args):
    image, word = ints(args.image), ints(args.word)
    if args.preflight:
        run_name = "runpod_preflight_ladder"
        model = MODEL_27B + "    max_new_tokens: 100\n    skip_capacity_probe: true\n"
        order = [("image", n) for n in image] + [("word", n) for n in word]
        # Largest first across both pair types, so an out-of-memory size shows up in the first minutes.
        order.sort(key=lambda pn: -pn[1] * (4 if pn[0] == "image" else 3))
        tasks = "".join(task_entry(pt, [n], ["inference"], 1) for pt, n in order)
        header = ("# Preflight for the 27B long-context ladder: one inference trial per size, largest first, free\n"
                  "# text on. Read by runpod/ladder_preflight.py, which trims the ladder to the sizes that fit.\n")
    else:
        run_name = "assoc_ladder_27b"
        model = (MODEL_27B + "    max_new_tokens: 100\n    skip_capacity_probe: true\n    counterbalance_sides: true\n"
                 + (f"    free_text_trials: {args.free_text_trials}\n" if args.free_text_trials >= 0 else ""))
        tasks = ""
        for n in sorted(set(image) | set(word)):
            for pt, sizes in (("image", image), ("word", word)):
                if n in sizes:
                    tasks += task_entry(pt, [n], ["inference", "direct_ab", "direct_bc"], args.trials)
        header = ("# Task 2: the 27B on associative inference from 2 chains to the context limit.\n"
                  "# One layout (cue_turn: the 27B passes the gate in all three), all three probes, answer sides\n"
                  "# counterbalanced within each cell (the 27B says \"2\" on ~70% of inference trials, so unbalanced\n"
                  "# sides move raw accuracy), free text only on the first trials of each cell (the score never\n"
                  "# reads it; it is the readability check), no capacity probe (the preflight ran the largest cells).\n"
                  "# Smallest size first, both pair types per size, so a stopped run leaves a complete curve.\n"
                  "# Run with: python runpod/run_bench_ext.py <this file>\n")
    return header, run_name, model, tasks


def think(args):
    pair_types = [p for p in args.pair_types.split(",") if p]
    chains = ints(args.chains)
    probes = [p for p in args.probes.split(",") if p]
    model = (MODEL_27B + "    max_new_tokens: 100\n    scoring: think\n    skip_capacity_probe: true\n"
             f"    loop_stop: {'false' if args.no_loop_stop else 'true'}\n"
             f"    thinking:\n      enabled: true\n      budget: {args.budget}\n      temperature: 0.6\n"
             "      top_p: 0.95\n      top_k: 20\n")
    if args.preflight:
        run_name = "runpod_preflight_think"
        tasks = "".join(task_entry(pt, [max(chains)], ["inference"], 2) for pt in pair_types)
        header = "# Preflight for the 27B thinking run: two inference trials at the largest chain count.\n"
    else:
        run_name = "assoc_think_27b"
        tasks = "".join(task_entry(pt, chains, probes, args.trials) for pt in pair_types)
        header = ("# Task 3: the 27B on the associative-inference gate cells with thinking on.\n"
                  "# Thinking exactly as in the repo's thinking configs (its `thinking` block and think(): Qwen's\n"
                  "# sampling, seeded per trial, budget as below); the forced choice and the free reply both follow\n"
                  "# the one trace. Added: loop_stop cuts a trace that repeats itself (recorded as stop \"loop\").\n"
                  "# Same cells, seed and episodes as the thinking-off run in results/assoc_inf_ab (cue_turn),\n"
                  "# no side counterbalancing, so the two compare trial for trial (runpod/assoc_think_report.py).\n"
                  "# Run with: python runpod/run_bench_ext.py <this file>\n")
    return header, run_name, model, tasks


def twins(args):
    pair_types = [p for p in args.pair_types.split(",") if p]
    chains = ints(args.chains)
    model = MODEL_27B + "    max_new_tokens: 100\n"
    tasks = "".join(task_entry(pt, chains, ["inference", "direct_ab", "direct_bc"], args.trials) for pt in pair_types)
    header = ("# The 27B's thinking-off gate cells (cue_turn), as in configs/assoc_inf_ab.yaml, for pairing with the\n"
              "# thinking run. Same run name, so restored cells are skipped. Run with run_bench.py.\n")
    return header, "assoc_inf_ab", model, tasks


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("kind", choices=["ladder", "think", "twins"])
    parser.add_argument("--out", required=True)
    parser.add_argument("--preflight", action="store_true")
    parser.add_argument("--trials", type=int, default=50)
    parser.add_argument("--image", default="2,5,25,100,250,500,800")
    parser.add_argument("--word", default="2,5,25,100,250,500,800,1200")
    parser.add_argument("--free-text-trials", type=int, default=5, help="-1 = free text on every trial")
    parser.add_argument("--pair-types", default="image,word")
    parser.add_argument("--chains", default="2,5,25")
    parser.add_argument("--probes", default="inference,direct_ab,direct_bc")
    parser.add_argument("--budget", type=int, default=12288)
    parser.add_argument("--no-loop-stop", action="store_true")
    args = parser.parse_args()

    header, run_name, model, tasks = {"ladder": ladder, "think": think, "twins": twins}[args.kind](args)
    if not tasks:
        raise SystemExit("no sizes left to run")
    text = f"{header}\nrun_name: {run_name}\n\noutput_dir: results\n\nseeds: [0]\n\nmodels:\n{model}\ntasks:\n{tasks}"
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(text)
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/ladder_preflight.py
#!/usr/bin/env python
"""Read the long-context preflight: which sizes fit, how long a trial takes, what the full ladder costs.

Usage (from the repo root):
    python runpod/ladder_preflight.py <preflight config> <ladder config> [--capacity GB] [--price $/h]
                                      [--json out.json]

The preflight runs one inference trial at the largest sizes (free text on, so the reply can be read).
Per pair type it reports each size as OK or FAILED (error, skipped, unreadable reply, context too long,
or within 5% of GPU memory), and the largest size that is OK. Sizes below the largest OK one are taken
to fit (memory grows with the prompt). The time estimate scales each measured trial's prefill time
linearly with prompt length down to the smaller sizes, which over-estimates them (attention costs grow
faster than linearly), so treat it as an upper bound. Free text is only generated on the first
`free_text_trials` trials of each ladder cell, so those trials are charged a second prefill.
--json writes {pair_type: {"max_ok": n_chains or null, "sizes": {...}}, "hours": h, "dollars": d}.
"""
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from assoc_preflight import readable  # noqa: E402
from bench.config import expand_runs, load_config  # noqa: E402


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("preflight_config")
    parser.add_argument("ladder_config")
    parser.add_argument("--capacity", type=float, default=0.0)
    parser.add_argument("--price", type=float, default=0.0)
    parser.add_argument("--json")
    args = parser.parse_args()

    pre = load_config(args.preflight_config)
    out, measured = {}, {}
    print("Preflight: one inference trial at the largest sizes, free text on.\n")
    for run in expand_runs(pre):
        c = run.condition
        pt, n = c["pair_type"], c["n_chains"]
        path = Path(pre["output_dir"]) / pre["run_name"] / run.task / run.result_name
        entry = {"ok": False, "reason": None}
        if not path.exists():
            entry["reason"] = "no result file (the job did not reach this cell)"
        else:
            d = json.loads(path.read_text())
            trials = d.get("trials") or []
            if d.get("error"):
                last = d["error"].strip().splitlines()[-1]
                entry["reason"] = ("out of GPU memory" if "out of memory" in last.lower() or "OutOfMemory" in last
                                   else last[:200])
            elif d.get("skipped"):
                entry["reason"] = f"skipped: {d['skipped']}"
            elif not trials:
                entry["reason"] = "no trials"
            else:
                t = trials[0]
                peak = d.get("peak_gpu_gb", 0)
                entry.update(prompt_tokens=t.get("prompt_tokens"), score_seconds=t.get("score_seconds"),
                             free_text_seconds=t.get("free_text_seconds"), peak_gb=peak,
                             choice_mass=round(t.get("choice_mass", 0), 3), reply=(t.get("free_response") or "")[:160])
                if not readable(t.get("free_response")):
                    entry["reason"] = "the reply is not readable text or a digit (possible gibberish at this length)"
                elif t.get("choice_mass", 0) < 0.5:
                    entry["reason"] = f"only {t.get('choice_mass', 0):.2f} of the probability is on the answer digits"
                elif args.capacity and peak > 0.95 * args.capacity:
                    entry["reason"] = f"peak {peak} GB is within 5% of the {args.capacity:.0f} GB available"
                else:
                    entry["ok"] = True
                    measured.setdefault(pt, []).append((n, t.get("prompt_tokens") or 0, t.get("score_seconds") or 0,
                                                        t.get("free_text_seconds") or 0))
        out.setdefault(pt, {"sizes": {}})["sizes"][n] = entry
        status = "OK    " if entry["ok"] else "FAILED"
        detail = (f"{entry.get('prompt_tokens')} tokens, {entry.get('score_seconds')} s to score, "
                  f"{entry.get('free_text_seconds')} s free text, peak {entry.get('peak_gb')} GB"
                  if entry.get("prompt_tokens") else "")
        print(f"  {pt:5s} {n:5d} chains  {status} {detail}" + (f"  <- {entry['reason']}" if entry["reason"] else ""))
        if entry.get("reply"):
            print(f"                     reply: {entry['reply']!r}")

    for pt, info in out.items():
        ok = sorted(n for n, e in info["sizes"].items() if e["ok"])
        failed = sorted(n for n, e in info["sizes"].items() if not e["ok"])
        # The largest OK size sets the top of the ladder; any failed size above it is dropped.
        info["max_ok"] = max(ok) if ok else None
        info["dropped"] = [n for n in failed if info["max_ok"] is None or n > info["max_ok"]]

    ladder = load_config(args.ladder_config)
    free_k = ladder["models"][0].get("free_text_trials")
    total_s = 0.0
    for run in expand_runs(ladder):
        c = run.condition
        pt, n = c["pair_type"], c["n_chains"]
        info = out.get(pt)
        if not info or info["max_ok"] is None or n > info["max_ok"] or pt not in measured:
            continue
        # Nearest measured size at or above n, scaled down by chain count (prompt length is ~linear in it).
        ref = min((m for m in measured[pt] if m[0] >= n), key=lambda m: m[0], default=max(measured[pt]))
        scale = n / ref[0]
        trials = c["n_trials"]
        free = trials if free_k is None else min(free_k, trials)
        total_s += scale * (trials * ref[2] + free * ref[3])
    load_s = 300                              # model load and kernel warm-up, once
    hours = (total_s + load_s) / 3600
    summary = {**out, "hours": round(hours, 1), "dollars": round(hours * args.price, 0) if args.price else None}
    print()
    for pt, info in out.items():
        if info["max_ok"] is None:
            print(f"{pt}: nothing fit. The ladder for {pt} pairs needs a bigger GPU (or two: set GPUS = \"0,1\").")
        elif info["dropped"]:
            print(f"{pt}: runs up to {info['max_ok']} chains; dropped {info['dropped']} (see above).")
        else:
            print(f"{pt}: every tested size fits; runs up to {info['max_ok']} chains.")
    cost = f", about ${hours * args.price:.0f} at ${args.price:.2f}/h" if args.price else ""
    print(f"\nEstimated full ladder: up to ~{hours:.1f} h{cost} (an upper bound: small sizes are scaled linearly).")
    if args.json:
        Path(args.json).write_text(json.dumps(summary, indent=2, default=str))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/assoc_ladder_report.py
#!/usr/bin/env python
"""Task 2 report: associative inference on the 27B from 2 chains to the context limit.

Usage (from the repo root):
    python runpod/assoc_ladder_report.py results/assoc_ladder_27b [--report out.md] [--csv out.csv]
                                         [--plot out.png]

One row per (pair type, probe, chains): trials, accuracy with its 95% Wilson interval, bias-corrected
(balanced) accuracy, the share of "2" answers, the mean prompt length in tokens, seconds per trial,
peak GPU memory, how many of the free-text replies read as text, and the primacy split (balanced
accuracy when the correct item was studied before vs after the foil). With counterbalanced sides
(the ladder config sets counterbalance_sides), raw and balanced accuracy are the same number, which
the report checks. Then a reading: for each probe, the largest size still above chance.

The plot puts accuracy against prompt length in tokens (log axis), one panel per pair type, one line
per probe, chance dashed, the chain counts written along the inference line.
"""
import argparse
import csv
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from assoc_preflight import readable  # noqa: E402
from src.metrics import wilson_ci  # noqa: E402

PROBES = ("direct_ab", "direct_bc", "inference")
PROBE_LABEL = {"direct_ab": "Direct A→B (studied)", "direct_bc": "Direct B→C (studied)",
               "inference": "Inference A→C (never studied)"}
PROBE_STYLE = {"inference": ("#2a78d6", "D"), "direct_ab": ("#eb6834", "s"), "direct_bc": ("#1baf7a", "^")}
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"


def side_acc(trials, side):
    sub = [t for t in trials if t["target"] == side]
    return sum(int(t["correct"]) for t in sub) / len(sub) if sub else float("nan")


def balanced(trials):
    return (side_acc(trials, 1) + side_acc(trials, 2)) / 2


def foil_chain(t):
    md = t.get("metadata", {})
    if "foil_option" in md:
        return md["foil_option"].get("chain_index")
    return md.get("foil_chain_index")


def load(run_dir):
    cells, errors = {}, []
    for path in sorted(Path(run_dir).glob("associative_inference/*.json")):
        d = json.loads(path.read_text())
        c = d["run"]["condition"]
        key = (c["pair_type"], c["probe"], c["n_chains"])
        if d.get("error") or d.get("skipped"):
            last = (d.get("error") or "").strip().splitlines()[-1:] or [d.get("skipped")]
            errors.append((key, str(last[0])[:160]))
            continue
        cells[key] = d
    return cells, errors


def row_for(key, d):
    trials = d["trials"]
    n = len(trials)
    k = sum(int(t["correct"]) for t in trials)
    acc, lo, hi = wilson_ci(k, n)
    earlier = [t for t in trials if foil_chain(t) is not None and t["metadata"]["chain_index"] < foil_chain(t)]
    later = [t for t in trials if foil_chain(t) is not None and t["metadata"]["chain_index"] > foil_chain(t)]
    free = [t for t in trials if t.get("free_response") is not None]
    tokens = [t["prompt_tokens"] for t in trials if "prompt_tokens" in t]
    secs = [t["score_seconds"] for t in trials if "score_seconds" in t]
    return {
        "pair_type": key[0], "probe": key[1], "n_chains": key[2], "trials": n,
        "accuracy": acc, "ci95_low": lo, "ci95_high": hi, "balanced": balanced(trials),
        "share_2": sum(t["reported"] == 2 for t in trials) / n,
        "side1_share": sum(t["target"] == 1 for t in trials) / n,
        "prompt_tokens": sum(tokens) / len(tokens) if tokens else None,
        "sec_per_trial": sum(secs) / len(secs) if secs else d.get("elapsed_s", 0) / n,
        "peak_gb": d.get("peak_gpu_gb"),
        "free_text_readable": f"{sum(readable(t['free_response']) for t in free)}/{len(free)}",
        "mass": sum(t.get("choice_mass", 0) for t in trials) / n,
        "primacy_earlier": balanced(earlier) if len({t['target'] for t in earlier}) == 2 else float("nan"),
        "primacy_later": balanced(later) if len({t['target'] for t in later}) == 2 else float("nan"),
    }


def fmt(v, spec=".2f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "-"
    return format(v, spec)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    parser.add_argument("--report")
    parser.add_argument("--csv")
    parser.add_argument("--plot")
    args = parser.parse_args()

    cells, errors = load(args.run_dir)
    if not cells and not errors:
        print(f"No results in {args.run_dir} yet.")
        return
    rows = [row_for(k, d) for k, d in sorted(cells.items(), key=lambda kv: (kv[0][0], PROBES.index(kv[0][1]), kv[0][2]))]
    out = []

    def say(line=""):
        print(line)
        out.append(line)

    model = next(iter(cells.values()))["run"]["model"]["name"] if cells else "?"
    say(f"# Associative inference to the context limit: {model}")
    say()
    say(f"`{args.run_dir}`: one layout (cue_turn), answer sides counterbalanced within each cell, 50 trials "
        "per cell unless stated. Accuracy with a 95% Wilson interval; chance is 0.50.")
    if errors:
        say()
        say(f"**{len(errors)} cells failed or were skipped:**")
        for key, msg in errors:
            say(f"- {key[0]} {key[1]} {key[2]} chains: {msg}")
    unbalanced = [r for r in rows if abs(r["accuracy"] - r["balanced"]) > 0.02]
    if unbalanced:
        say()
        say(f"Note: {len(unbalanced)} cells have raw and balanced accuracy more than 2 points apart, so their "
            "sides were not counterbalanced; read the balanced column for those.")

    say()
    say("## Reading")
    say()
    for pt in sorted({r["pair_type"] for r in rows}):
        for probe in PROBES:
            sub = sorted((r for r in rows if r["pair_type"] == pt and r["probe"] == probe), key=lambda r: r["n_chains"])
            if not sub:
                continue
            above = [r["n_chains"] for r in sub if r["ci95_low"] > 0.5]
            top = sub[-1]
            last = (f"still above chance at the largest size, {top['n_chains']} chains "
                    f"(~{fmt(top['prompt_tokens'] / 1000 if top['prompt_tokens'] else None, '.0f')}k tokens)"
                    if top["ci95_low"] > 0.5 else
                    f"above chance up to {max(above)} chains" if above else "not above chance at any size")
            curve = ", ".join(f"{r['n_chains']}: {r['accuracy']:.2f}" for r in sub)
            say(f"- {pt} pairs, {probe}: {last}. ({curve})")
    say()
    say("## Every cell")
    say()
    say("| pair | probe | chains | tokens | acc [95% CI] | balanced | says \"2\" | primacy: before / after | "
        "free text readable | s/trial | peak GB |")
    say("|---|---|---|---|---|---|---|---|---|---|---|")
    for r in rows:
        say(f"| {r['pair_type']} | {r['probe']} | {r['n_chains']} | {fmt(r['prompt_tokens'], ',.0f')} | "
            f"{fmt(r['accuracy'])} [{fmt(r['ci95_low'])}, {fmt(r['ci95_high'])}] | {fmt(r['balanced'])} | "
            f"{fmt(r['share_2'])} | {fmt(r['primacy_earlier'])} / {fmt(r['primacy_later'])} | "
            f"{r['free_text_readable']} | {fmt(r['sec_per_trial'], '.1f')} | {fmt(r['peak_gb'], '.1f')} |")
    say()
    say("balanced = mean of the accuracies on side-1 and side-2 trials. primacy = balanced accuracy when the "
        "correct item was studied before / after the foil. free text = the replies generated (first trials of "
        "each cell only), counted readable if they are a digit or English text.")

    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")
    if args.csv and rows:
        with open(args.csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
        print(f"wrote {args.csv}")
    if args.plot and rows:
        plot(rows, Path(args.plot), model)
        print(f"wrote {args.plot}")


def plot(rows, path, model):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    pair_types = sorted({r["pair_type"] for r in rows})
    fig, axes = plt.subplots(1, len(pair_types), figsize=(7 * len(pair_types), 5.2), squeeze=False, sharey=True)
    for ax, pt in zip(axes[0], pair_types):
        for probe in PROBES:
            sub = sorted((r for r in rows if r["pair_type"] == pt and r["probe"] == probe and r["prompt_tokens"]),
                         key=lambda r: r["n_chains"])
            if not sub:
                continue
            color, marker = PROBE_STYLE[probe]
            xs = [r["prompt_tokens"] for r in sub]
            ys = [100 * r["accuracy"] for r in sub]
            err = [[max(0.0, y - 100 * r["ci95_low"]) for y, r in zip(ys, sub)],
                   [max(0.0, 100 * r["ci95_high"] - y) for y, r in zip(ys, sub)]]
            ax.errorbar(xs, ys, yerr=err, color=color, marker=marker, ms=7, lw=2, capsize=3, mec="white", mew=1.2,
                        label=PROBE_LABEL[probe])
            if probe == "inference":
                for x, y, r in zip(xs, ys, sub):
                    ax.annotate(f"{r['n_chains']}", (x, y), xytext=(0, -16), textcoords="offset points",
                                ha="center", fontsize=8, color=MUTED)
        ax.axhline(50, ls="--", color="#999999", lw=1)
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:.0f}k" if v >= 1000 else f"{v:.0f}"))
        ax.set_ylim(0, 104)
        ax.set_xlabel("Prompt length (tokens, log scale); grey numbers = chains studied", color=INK)
        ax.set_title(f"{pt.capitalize()} pairs (C is {'an image' if pt == 'image' else 'a word'})", fontsize=11, color=INK)
        ax.grid(color=GRID, lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0][0].set_ylabel("Accuracy (%), 95% Wilson CI", color=INK)
    axes[0][0].legend(frameon=False, fontsize=9, loc="lower left")
    fig.suptitle(f"{model} associative inference vs context length (one layout, counterbalanced sides; dashed = chance)",
                 color=INK)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/assoc_preflight.py
#!/usr/bin/env python
"""Check the associative-inference preflights and estimate the full runs.

Usage (from the repo root):
    python runpod/assoc_preflight.py --check <preflight config>=<full config> [...]
        [--capacity GB] [--price DOLLARS_PER_HOUR] [--json out.json]

For each model in each preflight config: every preflight cell exists with no error or skip, the forced
choice holds real probability mass (mean choice_mass >= 0.8), the free replies are readable text (a
digit, or English sentences, not token soup), and peak GPU memory stays under 95% of --capacity.
Reason-then-choose cells must also carry their reasoning stats (proof they ran through
run_bench_ext.py). Prints PASS/FAIL per model with the reasons, some replies, and a rough time for
that model's part of the full config: (its trials) x (seconds per trial at the preflight's size),
an upper bound since most full cells are smaller. --json writes {model: {"passed", "reasons", "seconds_per_trial"}}.
"""
import argparse
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402

COMMON = {"the", "a", "an", "is", "are", "was", "of", "and", "to", "in", "with", "image", "images", "cue", "pair",
          "pairs", "paired", "first", "second", "which", "this", "that", "it", "be", "on", "for", "as", "word",
          "words", "option", "answer", "correct", "therefore", "both", "not", "so", "based", "shows", "associated"}


def readable(reply):
    reply = (reply or "").strip()
    if reply.strip("*").rstrip(".") in ("1", "2") or re.fullmatch(r"\**[12]\**[.):]?\s+\**[A-Za-z][A-Za-z '-]*\**\.?", reply):
        return True                                       # "2" or "2. BLOUSE"
    if not reply:
        return False
    ascii_share = sum(ch.isascii() for ch in reply) / len(reply)
    words = re.findall(r"[A-Za-z]{1,}", reply.lower())
    common = sum(w in COMMON for w in words) / len(words) if words else 0
    return ascii_share >= 0.9 and len(words) >= 3 and common >= 0.15


def results(cfg):
    out = {}
    for run in expand_runs(cfg):
        path = Path(cfg["output_dir"]) / cfg["run_name"] / run.task / run.result_name
        out.setdefault(run.model["name"], []).append((run, json.loads(path.read_text()) if path.exists() else None))
    return out


def full_trials(cfg):
    trials = {}
    for run in expand_runs(cfg):
        trials[run.model["name"]] = trials.get(run.model["name"], 0) + run.condition.get("n_trials", 10)
    return trials


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--check", nargs="+", required=True, metavar="PREFLIGHT=FULL")
    parser.add_argument("--capacity", type=float, default=0.0, help="GB of GPU memory the jobs run on")
    parser.add_argument("--price", type=float, default=0.0)
    parser.add_argument("--json")
    args = parser.parse_args()

    verdicts, total_h = {}, 0.0
    for spec in args.check:
        pre_path, _, full_path = spec.partition("=")
        pre_cfg = load_config(pre_path)
        full = full_trials(load_config(full_path)) if full_path else {}
        print(f"===== {pre_path}" + (f"  (full run: {full_path})" if full_path else ""))
        for model, cells in results(pre_cfg).items():
            reasons, replies, ok_replies, masses, per_trial, peaks = [], [], 0, [], [], []
            rtc = pre_cfg_model(pre_cfg, model).get("scoring") in ("reason_then_choose", "think")
            for run, d in cells:
                c = run.condition
                label = f"{c['pair_type']} {c['probe']} {c['n_chains']} chains {c['prompt']}"
                if d is None:
                    reasons.append(f"{label}: no result file")
                    continue
                if d.get("error"):
                    reasons.append(f"{label}: {d['error'].strip().splitlines()[-1][:160]}")
                    continue
                if d.get("skipped"):
                    reasons.append(f"{label}: skipped ({d['skipped']})")
                    continue
                trials = d.get("trials") or []
                if not trials:
                    reasons.append(f"{label}: no trials")
                    continue
                peaks.append(d.get("peak_gpu_gb", 0))
                per_trial.append(d.get("elapsed_s", 0) / len(trials))
                for t in trials:
                    masses.append(t.get("choice_mass", 0))
                    replies.append((label, t))
                    ok_replies += readable(t.get("free_response"))
                    if rtc and "reasoning_tokens" not in t:
                        reasons.append(f"{label}: no reasoning stats, so it did not run through run_bench_ext.py")
                        break
            if masses and sum(masses) / len(masses) < 0.8:
                reasons.append(f"mean choice_mass {sum(masses) / len(masses):.2f} is under 0.8: "
                               "the answer digits are not where the probability is")
            if replies and ok_replies < len(replies) - 1:
                reasons.append(f"only {ok_replies} of {len(replies)} replies read as text or a digit (possible gibberish)")
            if args.capacity and peaks and max(peaks) > 0.95 * args.capacity:
                reasons.append(f"peak {max(peaks)} GB is within 5% of the {args.capacity:.0f} GB available")
            passed = not reasons
            verdicts[model] = {"passed": passed, "reasons": reasons}

            peak = f"peak {max(peaks):.1f} GB" if peaks else "no peak recorded"
            line = f"{model:12s} {'PASS' if passed else 'FAIL'}   {peak}"
            if per_trial and model in full:
                # Thinking traces vary a lot in length, so the mean of the trials run, not the fastest cell.
                secs = sum(per_trial) / len(per_trial) if rtc else min(per_trial)
                hours = full[model] * secs / 3600
                total_h += hours
                verdicts[model]["seconds_per_trial"] = round(secs, 1)
                line += f"   ~{secs:.1f} s/trial -> full run ~{hours:.1f} h at most ({full[model]} trials)"
            print(line)
            for r in reasons:
                print("     -", r)
            for label, t in replies[:3]:
                extra = (f" tokens={t['reasoning_tokens']} finished={t['reasoning_finished']} "
                         f"stated={t.get('stated_answer')}") if "reasoning_tokens" in t else ""
                if "thinking_end" in t:
                    extra += f" thinking_end={t['thinking_end']}"
                if "prompt_tokens" in t:
                    extra += f" prompt_tokens={t['prompt_tokens']}"
                print(f"     [{label}] target={t['target']} choice={t['reported']} mass={t.get('choice_mass', 0):.3f}{extra}")
                if "thinking" in t:
                    trace = t["thinking"]["trace"]
                    print(f"        thinking: {trace[:300]!r}" + (f" ... {trace[-200:]!r}" if len(trace) > 500 else ""))
                reply = t.get("free_response") or ""
                print(f"        reply: {reply[:300]!r}" + (f" ... {reply[-200:]!r}" if len(reply) > 500 else ""))
        print()
    if total_h:
        cost = f", about ${total_h * args.price:.0f} at ${args.price:.2f}/h" if args.price else ""
        print(f"Rough total for the models above: up to ~{total_h:.1f} h{cost}, plus model loading (a few minutes each).")
    if args.json:
        Path(args.json).write_text(json.dumps(verdicts, indent=2))


def pre_cfg_model(cfg, name):
    return next(m for m in cfg["models"] if m["name"] == name)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/assoc_think_report.py
#!/usr/bin/env python
"""Task 3 report: the 27B on associative inference with Qwen's thinking mode vs without, trial for trial.

Usage (from the repo root):
    python runpod/assoc_think_report.py results/assoc_inf_ab results/assoc_think_27b [--model qwen3.5-27b]
                                        [--report out.md] [--csv out.csv] [--plot out.png]

The thinking run uses the standard run's cells (cue_turn layout, 2/5/25 chains), seed and episodes,
without side counterbalancing, so trial i of a cell is the same episode and probe in both. That is
checked: a cell is paired only if every trial's target and tested chain match.

Per cell and pooled over pair types (per probe and chain count): raw and bias-corrected (balanced)
accuracy for each, the trials gained and lost with an exact McNemar test, the share of "2" answers
(the 27B's side bias on the inference question), and for the thinking run how the thinking ended
(closed by the model, stopped as a loop, out of budget) and its mean length. The question the run
answers is whether thinking helps the inference step, in particular at 2 chains.
"""
import argparse
import csv
import math
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from assoc_reasoned_report import load, mcnemar_p  # noqa: E402
from src.metrics import wilson_ci  # noqa: E402

PROBES = ("direct_ab", "direct_bc", "inference")
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"
STD_STYLE, THINK_STYLE = ("#777777", "o"), ("#2a78d6", "D")


def side_acc(trials, side):
    sub = [t for t in trials if t["target"] == side]
    return sum(int(t["correct"]) for t in sub) / len(sub) if sub else float("nan")


def balanced(trials):
    return (side_acc(trials, 1) + side_acc(trials, 2)) / 2


def balanced_ci(trials, reps=2000):
    rng = random.Random(0)
    vals = []
    for _ in range(reps):
        s = rng.choices(trials, k=len(trials))
        if len({t["target"] for t in s}) == 2:
            vals.append(balanced(s))
    vals.sort()
    return (vals[int(0.025 * len(vals))], vals[int(0.975 * len(vals)) - 1]) if vals else (float("nan"),) * 2


def compare(std, think):
    common = sorted(set(std) & set(think))
    paired = bool(common) and all(std[k]["target"] == think[k]["target"]
                                  and std[k]["metadata"].get("chain_index") == think[k]["metadata"].get("chain_index")
                                  for k in common)
    s, t = list(std.values()), list(think.values())
    row = {"paired": paired, "n": len(t)}
    for name, trials in (("std", s), ("think", t)):
        k = sum(int(x["correct"]) for x in trials)
        acc, lo, hi = wilson_ci(k, len(trials)) if trials else (float("nan"),) * 3
        row.update({f"{name}_acc": acc, f"{name}_bal": balanced(trials) if trials else float("nan"),
                    f"{name}_says2": sum(x["reported"] == 2 for x in trials) / len(trials) if trials else float("nan")})
    if paired:
        b = sum(not std[k]["correct"] and think[k]["correct"] for k in common)
        c = sum(std[k]["correct"] and not think[k]["correct"] for k in common)
        row.update(gained=b, lost=c, mcnemar_p=mcnemar_p(b, c))
    else:
        row.update(gained=None, lost=None, mcnemar_p=None)
    if t and "thinking_end" in t[0]:
        for end in ("closed", "loop", "budget", "eos"):
            row[f"end_{end}"] = sum(x["thinking_end"] == end for x in t) / len(t)
        row["think_tokens"] = sum(x["reasoning_tokens"] for x in t) / len(t)
        stated = [x for x in t if x.get("stated_answer") is not None]
        row["stated"] = len(stated) / len(t)
        row["stated_agree"] = (sum(str(x["stated_answer"]) == str(x["reported"]) for x in stated) / len(stated)
                               if stated else None)
    return row


def fmt(v, spec=".2f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "-"
    return format(v, spec)


def fmt_p(p):
    return "-" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("std_dir")
    parser.add_argument("think_dir")
    parser.add_argument("--model", default="qwen3.5-27b")
    parser.add_argument("--report")
    parser.add_argument("--csv")
    parser.add_argument("--plot")
    args = parser.parse_args()

    std = {k: v for k, v in load(args.std_dir).items() if k[0] == args.model}
    think = {k: v for k, v in load(args.think_dir).items() if k[0] == args.model}
    if not think:
        print(f"No thinking results for {args.model} in {args.think_dir} yet.")
        return
    keys = sorted(k for k in think if k in std)
    missing = sorted(k for k in think if k not in std)
    out = []

    def say(line=""):
        print(line)
        out.append(line)

    say(f"# {args.model} associative inference: thinking on vs off")
    say()
    say(f"Off: `{args.std_dir}` (standard scoring, thinking off). On: `{args.think_dir}` (Qwen thinking mode, the "
        "answer read off the logits after </think>). Same cells, seed and episodes.")
    if missing:
        say()
        say(f"**{len(missing)} thinking cells have no thinking-off twin** (e.g. {missing[0]}): upload the results zip "
            "that holds results/assoc_inf_ab with the 27B cells.")
    if not keys:
        return

    rows = [{"pair_type": k[1], "probe": k[2], "n_chains": k[3], "prompt": k[4], **compare(std[k], think[k])}
            for k in keys]
    unpaired = [r for r in rows if not r["paired"]]
    say()
    say(f"{len(rows)} cells compared, {len(rows) - len(unpaired)} paired trial for trial"
        + (f"; **{len(unpaired)} unpaired**, compared as independent samples" if unpaired else "") + ".")

    pooled = []
    say()
    say("## Pooled over pair types")
    say()
    say("| probe | chains | off: acc / balanced | on: acc / balanced [95% CI] | gained | lost | McNemar p | "
        "says \"2\" off → on | thinking closed / loop / budget | thinking tokens |")
    say("|---|---|---|---|---|---|---|---|---|---|")
    for probe in PROBES:
        for n in sorted({k[3] for k in keys}):
            sub = [k for k in keys if k[2] == probe and k[3] == n]
            if not sub:
                continue
            s_all = {(k, i): v for k in sub for i, v in std[k].items()}
            t_all = {(k, i): v for k in sub for i, v in think[k].items()}
            r = compare(s_all, t_all)
            lo, hi = balanced_ci(list(t_all.values()))
            pooled.append({"probe": probe, "n_chains": n, **r, "think_bal_low": lo, "think_bal_high": hi,
                           "std_bal_ci": balanced_ci(list(s_all.values()))})
            say(f"| {probe} | {n} | {fmt(r['std_acc'])} / {fmt(r['std_bal'])} | {fmt(r['think_acc'])} / "
                f"{fmt(r['think_bal'])} [{fmt(lo)}, {fmt(hi)}] | {fmt(r['gained'], 'd')} | {fmt(r['lost'], 'd')} | "
                f"{fmt_p(r['mcnemar_p'])} | {fmt(r['std_says2'])} → {fmt(r['think_says2'])} | "
                f"{fmt(r.get('end_closed'))} / {fmt(r.get('end_loop'))} / {fmt(r.get('end_budget'))} | "
                f"{fmt(r.get('think_tokens'), '.0f')} |")

    say()
    say("## Reading")
    say()
    for r in pooled:
        if r["probe"] != "inference":
            continue
        change = ("no reliable change" if r["mcnemar_p"] is None or r["mcnemar_p"] >= 0.05
                  else "a reliable gain" if r["gained"] > r["lost"] else "a reliable loss")
        verdict = "above chance" if r["think_bal_low"] > 0.5 else "at chance" if r["think_bal_high"] >= 0.5 else "below chance"
        say(f"- Inference at {r['n_chains']} chains: balanced {fmt(r['std_bal'])} off, {fmt(r['think_bal'])} on "
            f"({verdict} with thinking; {change}, McNemar p {fmt_p(r['mcnemar_p'])}).")
    loops = [r for r in pooled if (r.get("end_loop") or 0) + (r.get("end_budget") or 0) > 0.2]
    if loops:
        say(f"- In {len(loops)} probe/size groups over 20% of the thinking looped or ran out of budget, so it never "
            "reached its own conclusion there; the answer was read after a forced </think>.")
    say("- Each trial's thinking is in its `thinking.trace` field (free_response is the answer written after it): "
        "read a few 2-chain inference traces to see what the model takes \"indirectly associated\" to mean.")

    say()
    say("## Every cell")
    say()
    say("| pair | probe | chains | off acc | on acc | off / on balanced | p | says \"2\" off → on | closed / loop / budget | "
        "tokens | states answer | choice = stated |")
    say("|---|---|---|---|---|---|---|---|---|---|---|---|")
    for r in rows:
        say(f"| {r['pair_type']} | {r['probe']} | {r['n_chains']} | {fmt(r['std_acc'])} | {fmt(r['think_acc'])} | "
            f"{fmt(r['std_bal'])} / {fmt(r['think_bal'])} | {fmt_p(r['mcnemar_p'])} | {fmt(r['std_says2'])} → "
            f"{fmt(r['think_says2'])} | {fmt(r.get('end_closed'))} / {fmt(r.get('end_loop'))} / "
            f"{fmt(r.get('end_budget'))} | {fmt(r.get('think_tokens'), '.0f')} | {fmt(r.get('stated'))} | "
            f"{fmt(r.get('stated_agree'))} |")

    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")
    if args.csv:
        with open(args.csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
        print(f"wrote {args.csv}")
    if args.plot:
        plot(pooled, Path(args.plot), args.model)
        print(f"wrote {args.plot}")


def plot(pooled, path, model):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=True)
    labels = {"direct_ab": "Direct A→B (studied)", "direct_bc": "Direct B→C (studied)",
              "inference": "Inference A→C (never studied)"}
    for ax, probe in zip(axes, PROBES):
        sub = sorted((r for r in pooled if r["probe"] == probe), key=lambda r: r["n_chains"])
        xs = list(range(len(sub)))
        for key, (color, marker), lab, ci_key, dx in (
                ("std_bal", STD_STYLE, "Thinking off", "std_bal_ci", -0.06),
                ("think_bal", THINK_STYLE, "Thinking on", None, 0.06)):
            ys = [100 * r[key] for r in sub]
            if ci_key:
                cis = [r[ci_key] for r in sub]
            else:
                cis = [(r["think_bal_low"], r["think_bal_high"]) for r in sub]
            err = [[max(0.0, y - 100 * c[0]) for y, c in zip(ys, cis)], [max(0.0, 100 * c[1] - y) for y, c in zip(ys, cis)]]
            ax.errorbar([x + dx for x in xs], ys, yerr=err, color=color, marker=marker, ms=8, lw=2, capsize=3,
                        mec="white", mew=1.2, label=lab)
        ax.axhline(50, ls="--", color="#999999", lw=1)
        ax.set_xticks(xs)
        ax.set_xticklabels([str(r["n_chains"]) for r in sub])
        ax.set_xlim(-0.4, len(xs) - 0.6)
        ax.set_ylim(0, 104)
        ax.set_xlabel("Chains studied")
        ax.set_title(labels[probe], fontsize=11, color=INK)
        ax.grid(color=GRID, lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].set_ylabel("Balanced accuracy (%), 95% bootstrap CI")
    axes[0].legend(frameon=False, fontsize=9, loc="lower left")
    fig.suptitle(f"{model} associative inference, thinking off vs on (cue_turn layout, both pair types pooled; "
                 "dashed = chance)", color=INK)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/assoc_reasoned_report.py
#!/usr/bin/env python
"""Task 1 report: associative inference with standard vs reason-then-choose scoring, trial for trial.

Usage (from the repo root):
    python runpod/assoc_reasoned_report.py results/assoc_inf_ab results/assoc_inf_reasoned
        [--report out.md] [--csv out.csv] [--plot out.png]

Both runs use seed 0 and the same cells, so trial i of a cell sees the same episode and probe in
both. That is checked, not assumed: a cell is paired only if every trial's target and tested chain
match; otherwise it is compared unpaired and marked so.

Per cell: accuracy under each scoring, the change, an exact McNemar test on the discordant trials,
the BIAS flag (one answer on > 80% of trials) under each scoring, and for the reasoned run the mean
reply length, the share of replies cut off at the token limit, the share that state an answer, and
how often the forced choice agrees with the stated answer. Then the same pooled over cells, per model
and probe. Whether the reasoned run passes the repo's gate is assoc_gate.py's job (run it on the
reasoned directory); this report says whether and where the scoring changed the answers.
"""
import argparse
import csv
import json
import math
import sys
from collections import Counter, defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.metrics import wilson_ci  # noqa: E402

BIAS_MAX_FRAC = 0.80
CUT_OFF_WARN = 0.20
PROBES = ("direct_ab", "direct_bc", "inference")
PROBE_STYLE = {"inference": ("#2a78d6", "o"), "direct_ab": ("#eb6834", "s"), "direct_bc": ("#1baf7a", "^")}
INK, MUTED, GRID = "#222222", "#777777", "#dddddd"


def load(run_dir):
    """{(model, pair_type, probe, n_chains, prompt): {(seed, trial): trial}}"""
    cells = defaultdict(dict)
    for path in sorted(Path(run_dir).glob("associative_inference/*.json")):
        d = json.loads(path.read_text())
        if d.get("error") or d.get("skipped"):
            continue
        c = d["run"]["condition"]
        key = (d["run"]["model"]["name"], c["pair_type"], c["probe"], c["n_chains"], c["prompt"])
        for t in d.get("trials", []):
            cells[key][(d["run"]["seed"], t["trial"])] = t
    return cells


def mcnemar_p(b, c):
    """Exact two-sided McNemar p: binomial test of the discordant split against 1/2."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * tail)


def top_frac(trials):
    counts = Counter(str(t["reported"]) for t in trials)
    return counts.most_common(1)[0][1] / len(trials) if trials else 0.0


def bare(reply):
    return (reply or "").strip().rstrip(".") in ("1", "2")


def compare(plain, reasoned):
    """One row of numbers for a set of plain and reasoned trials (dicts keyed by (seed, trial))."""
    common = sorted(set(plain) & set(reasoned))
    paired = bool(common) and all(
        plain[k]["target"] == reasoned[k]["target"]
        and plain[k].get("metadata", {}).get("chain_index") == reasoned[k].get("metadata", {}).get("chain_index")
        for k in common)
    p_trials, r_trials = list(plain.values()), list(reasoned.values())
    row = {"paired": paired, "n_plain": len(p_trials), "n_reasoned": len(r_trials)}
    for name, trials in (("plain", p_trials), ("reasoned", r_trials)):
        k = sum(int(t["correct"]) for t in trials)
        acc, lo, hi = wilson_ci(k, len(trials)) if trials else (float("nan"),) * 3
        row.update({f"{name}_acc": acc, f"{name}_low": lo, f"{name}_high": hi,
                    f"{name}_top_frac": top_frac(trials),
                    f"{name}_bias": bool(trials) and top_frac(trials) > BIAS_MAX_FRAC})
    row["plain_long_replies"] = (sum(not bare(t.get("free_response")) for t in p_trials) / len(p_trials)
                                 if p_trials else None)
    if paired:
        b = sum(not plain[k]["correct"] and reasoned[k]["correct"] for k in common)
        c = sum(plain[k]["correct"] and not reasoned[k]["correct"] for k in common)
        row.update({"n_paired": len(common), "gained": b, "lost": c, "mcnemar_p": mcnemar_p(b, c),
                    "answer_changed": sum(plain[k]["reported"] != reasoned[k]["reported"] for k in common) / len(common)})
    else:
        row.update({"n_paired": 0, "gained": None, "lost": None, "mcnemar_p": None, "answer_changed": None})
    has_stats = [t for t in r_trials if "reasoning_tokens" in t]
    row["reasoning_recorded"] = len(has_stats) == len(r_trials) and bool(r_trials)
    if has_stats:
        stated = [t for t in has_stats if t.get("stated_answer") is not None]
        row.update({
            "mean_reasoning_tokens": sum(t["reasoning_tokens"] for t in has_stats) / len(has_stats),
            "cut_off": sum(not t["reasoning_finished"] for t in has_stats) / len(has_stats),
            "stated": len(stated) / len(has_stats),
            "stated_agree": (sum(str(t["stated_answer"]) == str(t["reported"]) for t in stated) / len(stated)
                             if stated else None),
        })
    return row


def fmt(v, spec=".2f"):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return "-"
    return format(v, spec)


def fmt_p(p):
    return "-" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("plain_dir")
    parser.add_argument("reasoned_dir")
    parser.add_argument("--report")
    parser.add_argument("--csv")
    parser.add_argument("--plot")
    args = parser.parse_args()

    plain, reasoned = load(args.plain_dir), load(args.reasoned_dir)
    keys = sorted(k for k in reasoned if k in plain)
    if not reasoned:
        print(f"No reasoned results in {args.reasoned_dir} yet.")
        return
    missing = sorted(k for k in reasoned if k not in plain)
    out = []

    def say(line=""):
        print(line)
        out.append(line)

    say("# Associative inference: standard vs reason-then-choose scoring")
    say()
    say(f"Standard: `{args.plain_dir}` (reply capped at 100 tokens, choice read with no reasoning in context). "
        f"Reasoned: `{args.reasoned_dir}` (reply up to 512 tokens, choice read after the reply). Same episodes, seed 0.")
    if missing:
        say()
        say(f"**{len(missing)} reasoned cells have no standard twin** (e.g. {' '.join(map(str, missing[0]))}). "
            "Upload the earlier results zip with results/assoc_inf_ab, or let the queue re-run configs/assoc_inf_ab.yaml.")
    if not keys:
        return

    rows = []
    for key in keys:
        model, pair_type, probe, n_chains, prompt = key
        rows.append({"model": model, "pair_type": pair_type, "probe": probe, "n_chains": n_chains,
                     "prompt": prompt, **compare(plain[key], reasoned[key])})

    unpaired = [r for r in rows if not r["paired"]]
    unrecorded = [r for r in rows if not r["reasoning_recorded"]]
    say()
    say(f"{len(rows)} cells compared; {len(rows) - len(unpaired)} paired trial for trial"
        + (f", **{len(unpaired)} unpaired** (episodes differ, compared as independent samples)" if unpaired else "")
        + ".")
    if unrecorded:
        say(f"**WARNING: {len(unrecorded)} reasoned cells carry no reasoning stats**, so they were not run with "
            "runpod/run_bench_ext.py and are standard scoring under another name. Do not use them.")

    # ---- pooled per model and probe ----
    say()
    say("## Pooled over layouts, pair types and chain counts")
    say()
    say("| model | probe | standard acc | reasoned acc | gained | lost | McNemar p | answers changed | "
        "reasoned BIAS cells | cut off | states answer | choice = stated |")
    say("|---|---|---|---|---|---|---|---|---|---|---|---|")
    pooled = []
    for model in sorted({r["model"] for r in rows}):
        for probe in PROBES:
            sub = [k for k in keys if k[0] == model and k[2] == probe]
            if not sub:
                continue
            p_all = {(k, t): v for k in sub for t, v in plain[k].items()}
            r_all = {(k, t): v for k in sub for t, v in reasoned[k].items()}
            row = compare(p_all, r_all)
            cell_rows = [r for r in rows if r["model"] == model and r["probe"] == probe]
            bias_cells = sum(r["reasoned_bias"] for r in cell_rows)
            pooled.append({"model": model, "probe": probe, **row})
            say(f"| {model} | {probe} | {fmt(row['plain_acc'])} | {fmt(row['reasoned_acc'])} "
                f"[{fmt(row['reasoned_low'])}, {fmt(row['reasoned_high'])}] | {fmt(row['gained'], 'd')} | "
                f"{fmt(row['lost'], 'd')} | {fmt_p(row['mcnemar_p'])} | {fmt(row['answer_changed'])} | "
                f"{bias_cells} of {len(cell_rows)} | {fmt(row.get('cut_off'))} | {fmt(row.get('stated'))} | "
                f"{fmt(row.get('stated_agree'))} |")

    # ---- reading ----
    say()
    say("## Reading")
    say()
    for r in pooled:
        if r["probe"] == "inference":
            continue
        verdict = ("above chance" if r["reasoned_low"] > 0.5 else "still at chance")
        change = ("no reliable change" if r["mcnemar_p"] is None or r["mcnemar_p"] >= 0.05
                  else "a reliable gain" if r["gained"] > r["lost"] else "a reliable loss")
        say(f"- {r['model']} {r['probe']}: {fmt(r['plain_acc'])} standard, {fmt(r['reasoned_acc'])} reasoned "
            f"({verdict} pooled; {change}, McNemar p {fmt_p(r['mcnemar_p'])}).")
    cut = [r for r in pooled if (r.get("cut_off") or 0) > CUT_OFF_WARN]
    if cut:
        say(f"- **{len(cut)} model/probe groups had over {CUT_OFF_WARN:.0%} of replies cut off at 512 tokens**, "
            "so the reasoning was often still unfinished: a longer limit is the next check before concluding.")
    else:
        say(f"- Replies mostly finished within 512 tokens (cut-off rate at most {CUT_OFF_WARN:.0%} everywhere), so "
            "truncation no longer explains the result.")
    say("- The gate itself (direct probes above chance with no BIAS at 2 and 5 chains, both models) is printed "
        "by assoc_gate.py on the reasoned run.")

    # ---- per cell ----
    say()
    say("## Every cell")
    say()
    say("| model | pair | probe | chains | layout | standard | reasoned | p | std top | rsn top | "
        "tokens | cut off | states | agree | paired |")
    say("|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|")
    for r in rows:
        say(f"| {r['model']} | {r['pair_type']} | {r['probe']} | {r['n_chains']} | {r['prompt']} | "
            f"{fmt(r['plain_acc'])} | {fmt(r['reasoned_acc'])} | {fmt_p(r['mcnemar_p'])} | "
            f"{fmt(r['plain_top_frac'])}{' B' if r['plain_bias'] else ''} | "
            f"{fmt(r['reasoned_top_frac'])}{' B' if r['reasoned_bias'] else ''} | "
            f"{fmt(r.get('mean_reasoning_tokens'), '.0f')} | {fmt(r.get('cut_off'))} | {fmt(r.get('stated'))} | "
            f"{fmt(r.get('stated_agree'))} | {'yes' if r['paired'] else 'NO'} |")
    say()
    say("top = share of the most common answer (B = BIAS, over 0.80). tokens = mean reasoned reply length. "
        "cut off = reply hit the 512-token limit. states = reply states an answer (\"the correct digit is 2\", "
        "or a bare digit). agree = forced choice matches the stated answer, of the replies that state one.")

    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")
    if args.csv:
        fields = list(rows[0])
        with open(args.csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            w.writerows(rows)
        print(f"wrote {args.csv}")
    if args.plot:
        plot(rows, Path(args.plot))
        print(f"wrote {args.plot}")


def plot(rows, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    models = sorted({r["model"] for r in rows}, key=lambda m: float(m.split("-")[-1].rstrip("b")))
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 5.2), squeeze=False)
    for ax, model in zip(axes[0], models):
        for probe in PROBES:
            sub = [r for r in rows if r["model"] == model and r["probe"] == probe]
            if not sub:
                continue
            color, marker = PROBE_STYLE[probe]
            ax.scatter([100 * r["plain_acc"] for r in sub], [100 * r["reasoned_acc"] for r in sub],
                       s=64, color=color, marker=marker, edgecolors="white", linewidths=1.5, label=probe, zorder=3)
        ax.plot([0, 100], [0, 100], color=MUTED, lw=1, zorder=1)
        ax.axhline(50, ls="--", color="#999999", lw=1, zorder=1)
        ax.axvline(50, ls="--", color="#999999", lw=1, zorder=1)
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_aspect("equal")
        ax.grid(color=GRID, lw=0.6, zorder=0)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        ax.set_xlabel("Standard scoring accuracy (%)", color=INK)
        ax.set_ylabel("Reason-then-choose accuracy (%)", color=INK)
        ax.set_title(f"{model}: one point per cell", fontsize=11, color=INK)
        ax.text(98, 3, "above the diagonal = reasoning helped", ha="right", fontsize=8, color=MUTED)
        ax.legend(frameon=False, fontsize=9, loc="upper left")
    fig.suptitle("Associative inference, standard vs reason-then-choose (dashed = chance)", color=INK)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/assoc_replies.py
#!/usr/bin/env python
"""Are the associative-inference answers real? One row per model (and per layout), from the trials.

Usage (from the repo root): python runpod/assoc_replies.py <run_dir> [--models a,b] [--report out.md]

check_results.py reads a reply's first "1" or "2" as its answer, which suits bare-digit replies but
misreads reasoning ("Image 1 shows a teddy bear...") as a disagreement. Here instead:
  readable   replies that are a digit, "2. BLOUSE", or English text (not token soup); GIBBERISH? under 0.9
  bare       replies that are just the digit
  states     replies that state an answer ("the correct digit is 2"); agree = forced choice matches it
  mass       mean probability on the two answer digits; LOW_MASS cells have over 10% of trials under 0.9
  BIAS cells one answer on over 80% of trials
"""
import argparse
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from assoc_preflight import readable  # noqa: E402
from run_bench_ext import stated_answer  # noqa: E402


def summarize(cells):
    trials = [t for c in cells for t in c["trials"]]
    n = len(trials)
    if not n:
        return None
    # Trials whose free text was skipped (free_text_trials) have no reply to judge.
    replied = [t for t in trials if t.get("free_response") is not None] or [{"free_response": ""}]
    stated = [(t, stated_answer(t.get("free_response"))) for t in replied]
    stated = [(t, s) for t, s in stated if s is not None]
    low_mass = bias = 0
    for c in cells:
        ts = c["trials"]
        if ts and sum(t.get("choice_mass", 0) < 0.9 for t in ts) / len(ts) > 0.10:
            low_mass += 1
        counts = defaultdict(int)
        for t in ts:
            counts[t["reported"]] += 1
        if ts and max(counts.values()) / len(ts) > 0.80:
            bias += 1
    return {
        "cells": len(cells), "trials": n,
        "readable": sum(readable(t.get("free_response")) for t in replied) / len(replied),
        "bare": sum((t.get("free_response") or "").strip().rstrip(".") in ("1", "2") for t in replied) / len(replied),
        "states": len(stated) / len(replied),
        "agree": sum(str(s) == str(t["reported"]) for t, s in stated) / len(stated) if stated else None,
        "mass": sum(t.get("choice_mass", 0) for t in trials) / n,
        "low_mass_cells": low_mass, "bias_cells": bias,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("run_dir")
    parser.add_argument("--models")
    parser.add_argument("--report")
    args = parser.parse_args()
    keep = set(args.models.split(",")) if args.models else None

    groups, errors = defaultdict(list), defaultdict(list)
    for path in sorted(Path(args.run_dir).glob("associative_inference/*.json")):
        d = json.loads(path.read_text())
        model = d["run"]["model"]["name"]
        if keep and model not in keep:
            continue
        if d.get("error") or d.get("skipped"):
            errors[model].append(path.name)
            continue
        cell = {"trials": d.get("trials") or []}
        groups[(model, "all")].append(cell)
        groups[(model, d["run"]["condition"]["prompt"])].append(cell)

    out = []

    def say(line=""):
        print(line)
        out.append(line)

    say(f"Replies in {args.run_dir}")
    say()
    say("| model | layout | cells | readable | bare digit | states answer | choice = stated | mass | LOW_MASS cells | BIAS cells | flags |")
    say("|---|---|---|---|---|---|---|---|---|---|---|")
    order = sorted(groups, key=lambda k: (float(k[0].split("-")[-1].rstrip("b")), k[1] != "all", k[1]))
    for key in order:
        s = summarize(groups[key])
        if s is None:
            continue
        flags = []
        if s["readable"] < 0.9:
            flags.append("GIBBERISH?")
        if key[1] == "all" and errors.get(key[0]):
            flags.append(f"{len(errors[key[0]])} ERROR/SKIPPED")
        agree = "-" if s["agree"] is None else f"{s['agree']:.2f}"
        say(f"| {key[0]} | {key[1]} | {s['cells']} | {s['readable']:.2f} | {s['bare']:.2f} | {s['states']:.2f} | "
            f"{agree} | {s['mass']:.3f} | {s['low_mass_cells']} | {s['bias_cells']} | {' '.join(flags)} |")
    for model, names in errors.items():
        say()
        say(f"{model}: {len(names)} cells failed or were skipped, e.g. {names[0]}")
    if args.report:
        Path(args.report).write_text("\n".join(out) + "\n")
        print(f"\nwrote {args.report}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/run_queue.py
#!/usr/bin/env python
"""Run benchmark jobs one after another, detached from the notebook.

Usage: python runpod/run_queue.py <queue.json>

queue.json holds {"name", "python", "repo", "log_dir", "env", "jobs": [{"config", "task", "gpus", "entry"}]}.
Each job is `python <entry> <config> [--task <task>]` with CUDA_VISIBLE_DEVICES set to its GPUs, where
entry is run_bench.py unless the job names another (runpod/run_bench_ext.py for reason-then-choose);
a job without a task runs every task in its config, loading the model once.
The status file <log_dir>/<name>.status.json is rewritten after every change, and each job gets
its own log. A failed job does not stop the queue. The runner skips result files that already
exist, so starting the same queue again resumes where it stopped.
"""
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path


def now():
    return datetime.now().isoformat(timespec="seconds")


def main():
    queue = json.loads(Path(sys.argv[1]).read_text())
    log_dir = Path(queue["log_dir"])
    log_dir.mkdir(parents=True, exist_ok=True)
    status_path = log_dir / f"{queue['name']}.status.json"
    status = {"name": queue["name"], "pid": os.getpid(), "started": now(), "finished": None,
              "jobs": [{**job, "state": "pending", "start": None, "end": None,
                        "returncode": None, "log": None} for job in queue["jobs"]]}

    def save():
        tmp = status_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(status, indent=2))
        tmp.replace(status_path)

    save()
    for i, job in enumerate(status["jobs"]):
        log = log_dir / f"{queue['name']}_{i:02d}_{Path(job['config']).stem}_{job.get('task') or 'all'}.log"
        job.update(state="running", start=now(), log=str(log))
        save()
        env = {**os.environ, **queue.get("env", {}), "CUDA_VISIBLE_DEVICES": str(job.get("gpus", "0"))}
        cmd = [queue["python"], job.get("entry") or "run_bench.py", job["config"]]
        if job.get("task"):
            cmd += ["--task", job["task"]]
        cmd += job.get("extra_args", [])
        with open(log, "a") as f:
            f.write(f"# {now()} {' '.join(cmd)}  (CUDA_VISIBLE_DEVICES={env['CUDA_VISIBLE_DEVICES']})\n")
            f.flush()
            returncode = subprocess.call(cmd, cwd=queue["repo"], env=env, stdout=f, stderr=subprocess.STDOUT)
        job.update(state="done" if returncode == 0 else "failed", end=now(), returncode=returncode)
        save()
    status["finished"] = now()
    save()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {REPO}/runpod/progress.py
#!/usr/bin/env python
"""Done / total result files for each config, using the runner's own run expansion.

Usage (from the repo root): python runpod/progress.py [--json] <config.yaml>[:<task>] [...]

bench.runner imports torch, so its done-test is repeated here to keep this check fast and usable
while a job holds the GPU. A result counts as done exactly as in bench.runner.is_done: the file
exists, parses, has no error, and was not skipped. Running this also validates each config's sizes
against the datasets (expand_runs raises on a size the data cannot supply), without loading a model.
--json prints one JSON list ({config, task, done, failed, skipped, todo, total} per spec) instead.
"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from bench.config import expand_runs, load_config  # noqa: E402


def state(path):
    if not path.exists():
        return "todo"
    try:
        result = json.loads(path.read_text())
    except json.JSONDecodeError:
        return "todo"
    if result.get("error") is not None:
        return "failed"
    if result.get("skipped"):
        return "skipped"
    return "done"


def main():
    args = sys.argv[1:]
    as_json = "--json" in args
    rows = []
    for spec in (a for a in args if a != "--json"):
        config_path, _, task = spec.partition(":")
        cfg = load_config(config_path)
        if task:
            cfg["tasks"] = [t for t in cfg["tasks"] if t["task"] == task]
        runs = expand_runs(cfg)
        states = [state(Path(cfg["output_dir"]) / cfg["run_name"] / r.task / r.result_name) for r in runs]
        counts = {s: states.count(s) for s in ("done", "failed", "skipped", "todo")}
        rows.append({"config": config_path, "task": task or None, **counts, "total": len(runs)})
        if not as_json:
            label = config_path + (f" --task {task}" if task else "")
            print(f"{label:58s} {counts['done']:4d}/{len(runs):<4d} done   "
                  f"failed={counts['failed']} skipped={counts['skipped']} todo={counts['todo']}")
    if as_json:
        print(json.dumps(rows))


if __name__ == "__main__":
    main()

In [ ]:
THINK = "configs/runpod/assoc_think_27b.yaml"
PREFLIGHT = "configs/runpod/preflight_think.yaml"
TWINS = "configs/runpod/assoc_inf_ab_27b_cue_turn.yaml"
RUN_NAME, PREFLIGHT_RUN = "assoc_think_27b", "runpod_preflight_think"
think_args = ["--chains", ",".join(map(str, CHAINS)), "--budget", THINK_BUDGET, "--probes", ",".join(PROBES),
              "--trials", N_TRIALS] + ([] if LOOP_STOP else ["--no-loop-stop"])
bench("runpod/make_configs.py", "think", "--out", THINK, "--pair-types", ",".join(PAIR_TYPES), *think_args)
for pt in PAIR_TYPES:      # one config per pair type, for running two queues side by side
    bench("runpod/make_configs.py", "think", "--out", THINK.replace(".yaml", f"_{pt}.yaml"), "--pair-types", pt,
          *think_args)
bench("runpod/make_configs.py", "think", "--preflight", "--out", PREFLIGHT, "--pair-types", ",".join(PAIR_TYPES),
      *think_args)
bench("runpod/make_configs.py", "twins", "--out", TWINS, "--pair-types", ",".join(PAIR_TYPES),
      "--chains", ",".join(map(str, CHAINS)), "--trials", N_TRIALS)
bench("runpod/progress.py", PREFLIGHT, THINK, TWINS)

## 8. Preflight (about 15 minutes)

Two inference trials at 25 chains per pair type, thinking on. The 27B passes if every cell ran, the answer digits hold most of the probability after the thinking, the thinking reads as text, and the cells carry their thinking stats. It prints how each trace ended (closed by the model, stopped as a loop, out of budget), a few traces, and the time for the full run.

In [ ]:
launch("preflight", [(PREFLIGHT, GPUS, EXT)])
wait("preflight", every=20)

In [ ]:
verdict_file = OUT / "think_preflight.json"
verdict_file.unlink(missing_ok=True)
bench("runpod/assoc_preflight.py", "--check", f"{PREFLIGHT}={THINK}", "--capacity", round(capacity(), 1),
      "--price", PRICE_PER_HOUR, "--json", verdict_file, check=False)
PREFLIGHT_OK = json.loads(verdict_file.read_text()) if verdict_file.exists() else {}
n_queues = 2 if (PARALLEL == 2 or (PARALLEL == "auto" and capacity() >= 125)) else 1
if n_queues == 2:
    print(f"\n{capacity():.0f} GB of GPU memory: the full run uses two queues side by side (one per pair type), "
          "so expect roughly 1.5-2x less time than the estimate above.")

# When resuming: how much is actually left. Cells with a finished result (restored from an uploaded
# qwen35_runpod_results_*.zip, or from an earlier launch on this volume) are skipped by the runner.
LEFT_CODE = '''
import json, sys
sys.path.insert(0, ".")
from pathlib import Path
from bench.config import load_config, expand_runs
cfg = load_config(sys.argv[1])
todo = total = 0
for r in expand_runs(cfg):
    n = r.condition.get("n_trials", 0)
    total += n
    p = Path(cfg.get("output_dir", "results")) / cfg["run_name"] / r.task / r.result_name
    try:
        d = json.loads(p.read_text())
        done = d.get("error") is None and not d.get("skipped")
    except (OSError, ValueError):
        done = False
    todo += 0 if done else n
print(todo, total)
'''
counted = subprocess.run([str(PY), "-c", LEFT_CODE, THINK], cwd=REPO, env=ENV, capture_output=True, text=True)
try:
    todo, total = map(int, counted.stdout.strip().splitlines()[-1].split())
    secs = PREFLIGHT_OK.get("qwen3.5-27b", {}).get("seconds_per_trial")
    if secs and todo < total:
        one = todo * secs / 3600
        lo, hi = (one / 2, one / 1.5) if n_queues == 2 else (one, one)
        cost = f", about ${lo * PRICE_PER_HOUR:.0f}-{hi * PRICE_PER_HOUR:.0f}" if PRICE_PER_HOUR else ""
        print(f"\nResuming: {total - todo} of {total} trials are already finished and will be skipped, so the estimate "
              f"above overstates it. Left: {todo} trials, about {lo:.0f}-{hi:.0f} h on this pod{cost} at the preflight's "
              f"pace ({secs:.0f} s per inference trial). Direct-probe trials may run shorter or longer than that: "
              "the first finished cells show the real pace.")
except (ValueError, IndexError):
    pass

## 9. Full run (background)

Any missing thinking-off cells first (standard scoring, `run_bench.py`), then the thinking run: one queue, or on a big card two queues side by side, one per pair type, each with half the CPU threads. You can close the browser. To stop, use the stop cell; launching again resumes.

In [ ]:
QUEUES = []
passed = PREFLIGHT_OK.get("qwen3.5-27b", {}).get("passed")
if not passed and not FORCE_MAIN:
    print("Not launching: the 27B did not pass the preflight.")
    for r in PREFLIGHT_OK.get("qwen3.5-27b", {}).get("reasons", ["no preflight result"]):
        print("   -", r)
else:
    twins = [(TWINS, GPUS, None)] if len(twins_here()) < 3 * len(CHAINS) * len(PAIR_TYPES) else []
    QUEUES = ["main"] if n_queues == 1 or len(PAIR_TYPES) < 2 else [f"think_{pt}" for pt in PAIR_TYPES]
    set_threads(len(QUEUES))
    if len(QUEUES) == 1:
        launch("main", twins + [(THINK, GPUS, EXT)])
    else:
        for i, pt in enumerate(PAIR_TYPES):
            launch(f"think_{pt}", (twins if i == 0 else []) + [(THINK.replace(".yaml", f"_{pt}.yaml"), GPUS, EXT)])

In [ ]:
for q in QUEUES:
    status(q)
    print()

In [ ]:
wait(*QUEUES, every=300)     # blocks until everything finishes; interrupt any time, the jobs keep running

In [ ]:
# Uncomment to stop (finished cells are kept; launching again resumes):
# for q in QUEUES: stop(q)

## 10. Results

Thinking on vs off, trial for trial: raw and bias-corrected accuracy, trials gained and lost (McNemar), the share of "2" answers, how the thinking ended and how long it was, pooled over pair types per probe and chain count, then per cell. The plot shows bias-corrected accuracy for both. Then a few 2-chain inference traces to read.

In [ ]:
plot = f"plots/{RUN_NAME}/{RUN_NAME}_vs_off.png"
bench("runpod/assoc_think_report.py", "results/assoc_inf_ab", f"results/{RUN_NAME}", "--report",
      OUT / "TASK3_THINKING_REPORT.md", "--csv", OUT / "task3_thinking_cells.csv", "--plot", plot, check=False)
if (REPO / plot).exists():
    display(Image(filename=str(REPO / plot)))

In [ ]:
# A few 2-chain inference traces, start and end: what does the model take "indirectly associated" to mean?
shown = 0
for f in sorted((REPO / f"results/{RUN_NAME}/associative_inference").glob("*probe-inference*.json")):
    d = json.loads(f.read_text())
    if d["run"]["condition"]["n_chains"] != min(CHAINS) or not d.get("trials"):
        continue
    for t in d["trials"][:3]:
        text = (t.get("thinking") or {}).get("trace") or ""
        print(f"--- {d['run']['condition']['pair_type']} pairs, trial {t['trial']}: target {t['target']}, "
              f"answered {t['reported']}, thinking ended: {t.get('thinking_end')}, {t.get('reasoning_tokens')} tokens, reply {t.get('free_response')!r}")
        print(text[:1200] + (f"\n   [...]\n{text[-600:]}" if len(text) > 1800 else ""))
        shown += 1
print(f"\n({shown} traces shown; every trial's thinking is in its result file, field thinking.trace; free_response is the answer written after it.)")

## 11. Package

One zip in `/workspace/outputs`: unzipped at the repo root it adds `results/assoc_think_27b/` (and any thinking-off cells this pod ran, in `results/assoc_inf_ab/`), the configs, scripts and the plot; reports, the preflight and logs go to `runpod_extras/`.

In [ ]:
BUNDLE_TAG = "thinking"
EXTRA_RESULTS = ["results/assoc_inf_ab"]

stamp = datetime.now().strftime("%Y%m%d_%H%M")
bundle = OUT / f"qwen35_runpod_results_{BUNDLE_TAG}_{stamp}.zip"
freeze = OUT / "pip_freeze.txt"
LIST_PACKAGES = ("import importlib.metadata as m; print(chr(10).join(sorted("
                 "str(d.metadata['Name']) + '==' + d.version for d in m.distributions())))")
r = subprocess.run([str(PY), "-c", LIST_PACKAGES], capture_output=True, text=True)
if r.returncode == 0:
    freeze.write_text(r.stdout)

into_repo = [f"results/{RUN_NAME}", "configs/runpod", "runpod", f"plots/{RUN_NAME}"] + EXTRA_RESULTS
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in into_repo:
        for f in sorted((REPO / rel).rglob("*")):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(REPO).as_posix())
    pre = REPO / "results" / PREFLIGHT_RUN
    for f in sorted(pre.rglob("*")):
        if f.is_file():
            z.write(f, f"runpod_extras/results/{PREFLIGHT_RUN}/{f.relative_to(pre).as_posix()}")
    for f in sorted(OUT.glob("*")):
        if f.is_file() and f.suffix in (".md", ".csv", ".json", ".txt") :
            z.write(f, f"runpod_extras/{f.name}")
    if (REPO / "RUNPOD_SOURCE.json").exists():
        z.write(REPO / "RUNPOD_SOURCE.json", "runpod_extras/RUNPOD_SOURCE.json")
    for f in sorted(LOGS.glob("*")):
        if f.is_file():
            z.write(f, f"runpod_extras/logs/{f.name}")

with zipfile.ZipFile(bundle) as z:
    names = z.namelist()
groups = {}
for n in names:
    parts = n.split("/")
    key = "/".join(parts[:2] if parts[0] in ("results", "runpod_extras", "plots") else parts[:1])
    groups[key] = groups.get(key, 0) + 1
print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB, {len(names)} files)")
for key, count in sorted(groups.items()):
    print(f"  {key:45s} {count:5d} files")
print("\nDownload it (right-click in the file browser, Download), then stop the pod.")

## Troubleshooting

- **"CUDA driver version is insufficient"**: the pod's driver is older than 570. Redeploy with CUDA 12.8 or newer.
- **The first trial is slow**: the linear-attention kernels compile on first use (1–3 minutes), then are cached in `/workspace/.triton`.
- **Everything is several times slower than the estimate and the GPU draws little power**: the jobs are starved on the CPU. Set `CPU_THREADS = 8` and launch again.
- **The pod restarted**: run the settings and helpers cells, then `status()`; if a queue is gone, run its launch cell again. Finished cells are kept and skipped.
- **The pod is gone**: start a new pod (ideally on the same network volume), upload the notebook, the repo zip and your latest results zip, and Run All. Finished cells are restored and skipped.
- **A job's full log**: the file named by `status()`, in `/workspace/runpod_logs`.